In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/BigContest_JSH/BigContest/data')

Mounted at /content/drive


In [2]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import time

import matplotlib.pyplot as plt

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
od = dd.read_csv('OD_all.csv', assume_missing = True, header = 0)


In [4]:
od.head()

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts
0,1.130564e+09,1.130560e+09,20230901.0,12:00,13:00,1.0,3.0,0.0,1.0,1.0,10869.0,58.0,7.0
1,2.714072e+09,2.714073e+09,20230901.0,12:00,12:00,1.0,4.0,0.0,0.0,0.0,2018.0,4.0,16.0
2,3.017056e+09,3.017059e+09,20230901.0,18:00,18:00,1.0,2.0,0.0,4.0,0.0,14070.0,39.0,8.0
3,2.917067e+09,2.917059e+09,20230901.0,19:00,19:00,0.0,4.0,0.0,0.0,0.0,2738.0,9.0,22.0
4,2.714058e+09,4.729025e+09,20230901.0,21:00,22:00,1.0,1.0,2.0,3.0,0.0,43707.0,85.0,10.0


In [5]:
modal_cd = {0: '차량',
            1: '시내버스',
            2: '지하철',
            3: '도보',
            4: '기타',
            5: '철도',
            6: '시외고속버스',
            7: '항공기',
            'nan': '-'}
od.modal = od['modal'].map(modal_cd)

In [6]:
purpose_cd = {0: '귀가',
              1: '업무',
              2: '학업',
              3: '쇼핑여가',
              4: '기타',
              5: '여행'}
od.origin_purpose = od['origin_purpose'].map(purpose_cd)
od.dest_purpose = od['dest_purpose'].map(purpose_cd)

In [7]:
use_columns = [
    'origin_hdong_cd', 'dest_hdong_cd', 'date', 'start_time', 'end_time',
    'gender', 'age', 'modal', 'origin_purpose', 'dest_purpose',
    'od_dist_avg', 'od_duration_avg', 'od_cnts'
]

In [8]:
dtype_dict = {
    'origin_hdong_cd': 'int64',
    'dest_hdong_cd': 'int64',
    'date': 'int64',
    'start_time': 'object',
    'end_time': 'object',
    'gender': 'int8',
    'age': 'int8',
    'modal': 'float64',
    'origin_purpose': 'float64',
    'dest_purpose': 'int8',
    'od_dist_avg': 'int32',
    'od_duration_avg': 'int32',
    'od_cnts': 'int32'
}

In [9]:
dest_hdong_map = {
    1156054000.0: '여의동',
    4575034000.0: '임실 성수면',
    5115057200.0: '강릉시 포남2동',
    5115058000.0: '강릉시 초당동',
    2635052000.0: '부산광역시 해운대구 우2동',
    3020055000.0: '대전광역시 유성구 도룡동'
}

In [10]:
dest_hdong_codes = list(dest_hdong_map.keys())

In [11]:
filtered_data = od[od['dest_hdong_cd'].isin(dest_hdong_codes)]

In [12]:
od_data_pd = filtered_data.compute()

In [13]:
od_data_pd['Destination'] = od_data_pd['dest_hdong_cd'].map(dest_hdong_map)

In [14]:
dtype_optimized = {
    'origin_hdong_cd': 'int64',
    'dest_hdong_cd': 'int64',
    'date': 'int64',
    'start_time': 'object',
    'end_time': 'object',
    'gender': 'int8',
    'age': 'int8',
    'modal': 'int64',              # 이미 결측치 처리 후 int64로 변환됨
    'origin_purpose': 'int64',     # 이미 결측치 처리 후 int64로 변환됨
    'dest_purpose': 'int8',
    'od_dist_avg': 'int32',
    'od_duration_avg': 'int32',
    'od_cnts': 'int32'
}

In [15]:
od_data = od.astype(dtype_optimized)

In [16]:
od_data_pd.origin_hdong_cd = od_data_pd.origin_hdong_cd.astype('str')
od_data_pd.dest_hdong_cd = od_data_pd.dest_hdong_cd.astype('str')
od_data_pd.date = od_data_pd.date.astype('int')
od_data_pd.modal = od_data_pd.modal.astype('str')
od_data_pd.origin_purpose = od_data_pd.origin_purpose.astype('str')
od_data_pd.dest_purpose = od_data_pd.dest_purpose.astype('str')

In [17]:
od.head()

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts
0,1.130564e+09,1.130560e+09,20230901.0,12:00,13:00,1.0,3.0,차량,업무,업무,10869.0,58.0,7.0
1,2.714072e+09,2.714073e+09,20230901.0,12:00,12:00,1.0,4.0,차량,귀가,귀가,2018.0,4.0,16.0
2,3.017056e+09,3.017059e+09,20230901.0,18:00,18:00,1.0,2.0,차량,기타,귀가,14070.0,39.0,8.0
3,2.917067e+09,2.917059e+09,20230901.0,19:00,19:00,0.0,4.0,차량,귀가,귀가,2738.0,9.0,22.0
4,2.714058e+09,4.729025e+09,20230901.0,21:00,22:00,1.0,1.0,지하철,쇼핑여가,귀가,43707.0,85.0,10.0


In [18]:
df = od_data_pd[od_data_pd['dest_purpose'].isin(['쇼핑여가', '여행', '기타'])]
df

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
1324,1114055000.0,1156054000.0,20230901,16:00,17:00,1.0,3.0,지하철,업무,쇼핑여가,28195.0,41.0,7.0,여의동
1363,3611055600.0,3020055000.0,20230901,15:00,15:00,0.0,0.0,차량,귀가,기타,43798.0,42.0,27.0,대전광역시 유성구 도룡동
2148,2823766000.0,1156054000.0,20230901,16:00,16:00,0.0,0.0,차량,귀가,쇼핑여가,55464.0,50.0,26.0,여의동
4181,1150061100.0,1156054000.0,20230901,12:00,13:00,1.0,0.0,차량,여행,여행,66357.0,52.0,19.0,여의동
4398,4119080000.0,1156054000.0,20230901,16:00,17:00,1.0,0.0,차량,귀가,쇼핑여가,14481.0,78.0,19.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
713795,5115056000.0,5115057200.0,20231015,17:00,17:00,1.0,3.0,시내버스,귀가,쇼핑여가,14336.0,9.0,5.0,강릉시 포남2동
714203,2623051000.0,2635052000.0,20231015,10:00,10:00,0.0,2.0,차량,여행,여행,23125.0,26.0,6.0,부산광역시 해운대구 우2동
714641,4511170300.0,4575034000.0,20231015,13:00,15:00,0.0,3.0,차량,귀가,기타,208055.0,91.0,5.0,임실 성수면
714799,5115066500.0,5115057200.0,20231015,13:00,15:00,1.0,1.0,차량,쇼핑여가,여행,320011.0,101.0,6.0,강릉시 포남2동


In [19]:
od_car = df[df['modal'] == '차량']
od_car

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
1363,3611055600.0,3020055000.0,20230901,15:00,15:00,0.0,0.0,차량,귀가,기타,43798.0,42.0,27.0,대전광역시 유성구 도룡동
2148,2823766000.0,1156054000.0,20230901,16:00,16:00,0.0,0.0,차량,귀가,쇼핑여가,55464.0,50.0,26.0,여의동
4181,1150061100.0,1156054000.0,20230901,12:00,13:00,1.0,0.0,차량,여행,여행,66357.0,52.0,19.0,여의동
4398,4119080000.0,1156054000.0,20230901,16:00,17:00,1.0,0.0,차량,귀가,쇼핑여가,14481.0,78.0,19.0,여의동
4648,1154551000.0,1156054000.0,20230901,17:00,18:00,0.0,3.0,차량,업무,쇼핑여가,50964.0,42.0,7.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
713687,5115057100.0,5115057200.0,20231015,20:00,20:00,0.0,2.0,차량,쇼핑여가,쇼핑여가,7792.0,16.0,5.0,강릉시 포남2동
714203,2623051000.0,2635052000.0,20231015,10:00,10:00,0.0,2.0,차량,여행,여행,23125.0,26.0,6.0,부산광역시 해운대구 우2동
714641,4511170300.0,4575034000.0,20231015,13:00,15:00,0.0,3.0,차량,귀가,기타,208055.0,91.0,5.0,임실 성수면
714799,5115066500.0,5115057200.0,20231015,13:00,15:00,1.0,1.0,차량,쇼핑여가,여행,320011.0,101.0,6.0,강릉시 포남2동


In [20]:
od_bus = df[df['modal'] == '시내버스']
od_bus

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
5032,1156051500.0,1156054000.0,20230901,13:00,13:00,1.0,2.0,시내버스,귀가,쇼핑여가,12739.0,19.0,7.0,여의동
5184,2635065000.0,2635052000.0,20230901,15:00,15:00,1.0,3.0,시내버스,귀가,쇼핑여가,9099.0,6.0,8.0,부산광역시 해운대구 우2동
24445,5115066500.0,5115058000.0,20230901,10:00,10:00,0.0,4.0,시내버스,여행,여행,16646.0,11.0,8.0,강릉시 초당동
27874,5115059000.0,5115057200.0,20230901,08:00,08:00,0.0,1.0,시내버스,귀가,기타,21231.0,16.0,65.0,강릉시 포남2동
57244,2635052500.0,2635052000.0,20230901,10:00,10:00,1.0,4.0,시내버스,귀가,쇼핑여가,10272.0,17.0,14.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694156,1153051000.0,1156054000.0,20231015,11:00,11:00,0.0,3.0,시내버스,귀가,쇼핑여가,22942.0,32.0,5.0,여의동
698767,1141058500.0,1156054000.0,20231015,15:00,16:00,1.0,3.0,시내버스,쇼핑여가,쇼핑여가,64945.0,56.0,5.0,여의동
704557,3020060000.0,3020055000.0,20231015,10:00,10:00,1.0,3.0,시내버스,여행,여행,17542.0,26.0,5.0,대전광역시 유성구 도룡동
712936,1159066000.0,1156054000.0,20231015,09:00,09:00,0.0,1.0,시내버스,귀가,기타,9126.0,18.0,6.0,여의동


In [21]:
od_subway = df[df['modal'] == '지하철']
od_subway

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
1324,1114055000.0,1156054000.0,20230901,16:00,17:00,1.0,3.0,지하철,업무,쇼핑여가,28195.0,41.0,7.0,여의동
104928,1114052000.0,1156054000.0,20230901,18:00,18:00,0.0,3.0,지하철,업무,쇼핑여가,30122.0,38.0,7.0,여의동
110058,1144056500.0,1156054000.0,20230901,08:00,08:00,1.0,2.0,지하철,귀가,기타,22246.0,24.0,7.0,여의동
138846,1168064000.0,1156054000.0,20230901,18:00,19:00,1.0,3.0,지하철,업무,쇼핑여가,57873.0,65.0,9.0,여의동
154913,1150060300.0,1156054000.0,20230901,08:00,09:00,1.0,2.0,지하철,귀가,기타,21049.0,35.0,7.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672282,1150053000.0,1156054000.0,20231015,11:00,12:00,0.0,2.0,지하철,귀가,쇼핑여가,22601.0,36.0,5.0,여의동
686534,1135069500.0,1156054000.0,20231015,10:00,12:00,0.0,3.0,지하철,기타,쇼핑여가,36423.0,90.0,5.0,여의동
689070,2635053000.0,2635052000.0,20231015,16:00,16:00,0.0,3.0,지하철,귀가,쇼핑여가,11429.0,26.0,6.0,부산광역시 해운대구 우2동
690438,1150053500.0,1156054000.0,20231015,14:00,14:00,1.0,3.0,지하철,귀가,쇼핑여가,11308.0,34.0,5.0,여의동


In [22]:
od_train= df[df['modal'] == '철도']
od_train

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
655811,1123057000.0,5115057200.0,20230901,11:00,14:00,0.0,2.0,철도,여행,여행,420543.0,150.0,5.0,강릉시 포남2동
16847,1123056000.0,5115057200.0,20230901,11:00,14:00,1.0,3.0,철도,기타,여행,387288.0,132.0,5.0,강릉시 포남2동
500009,1123056000.0,5115057200.0,20230901,21:00,23:00,1.0,3.0,철도,여행,여행,391050.0,123.0,5.0,강릉시 포남2동
580952,1123070500.0,5115057200.0,20230901,18:00,20:00,0.0,2.0,철도,기타,여행,396603.0,139.0,5.0,강릉시 포남2동
49677,1117053000.0,5115057200.0,20230901,09:00,12:00,1.0,4.0,철도,여행,여행,407603.0,190.0,5.0,강릉시 포남2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60994,1165058100.0,5115057200.0,20231015,18:00,22:00,1.0,4.0,철도,기타,기타,744523.0,200.0,5.0,강릉시 포남2동
194398,1123056000.0,5115057200.0,20231015,12:00,14:00,1.0,4.0,철도,기타,여행,613094.0,120.0,5.0,강릉시 포남2동
483869,3114059500.0,2635052000.0,20231015,11:00,13:00,0.0,0.0,철도,귀가,쇼핑여가,227785.0,151.0,13.0,부산광역시 해운대구 우2동
189744,1123056000.0,5115058000.0,20231015,10:00,12:00,0.0,1.0,철도,쇼핑여가,여행,339695.0,152.0,7.0,강릉시 초당동


In [23]:
# festival_periods = {
#     1156054000.0: ('20231007', '20231007'),              # 여의동: 2023-10-07
#     4575034000.0: ('20231006', '20231009'),              # 임실 성수면: 2023-10-06 ~ 2023-10-09
#     5115057200.0: ('20231012', '20231015'),              # 강릉시 포남2동: 2023-10-12 ~ 2023-10-15
#     5115058000.0: ('20231012', '20231015'),              # 강릉시 초당동: 2023-10-12 ~ 2023-10-15
#     2635052000.0: ('20231004', '20231013'),              # 부산광역시 해운대구 우2동: 2023-10-04 ~ 2023-10-13
#     3020055000.0: ('20231008', '20231010')               # 대전광역시 유성구 도룡동: 2023-10-08 ~ 2023-10-10
# }

# 여의동

차량 탄소발자국

In [24]:
df_s = od_car[od_car['dest_hdong_cd'] == '1156054000.0']
df_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
2148,2823766000.0,1156054000.0,20230901,16:00,16:00,0.0,0.0,차량,귀가,쇼핑여가,55464.0,50.0,26.0,여의동
4181,1150061100.0,1156054000.0,20230901,12:00,13:00,1.0,0.0,차량,여행,여행,66357.0,52.0,19.0,여의동
4398,4119080000.0,1156054000.0,20230901,16:00,17:00,1.0,0.0,차량,귀가,쇼핑여가,14481.0,78.0,19.0,여의동
4648,1154551000.0,1156054000.0,20230901,17:00,18:00,0.0,3.0,차량,업무,쇼핑여가,50964.0,42.0,7.0,여의동
6486,1156053500.0,1156054000.0,20230901,18:00,18:00,0.0,3.0,차량,기타,기타,17311.0,21.0,7.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707047,1153056000.0,1156054000.0,20231015,15:00,16:00,0.0,4.0,차량,쇼핑여가,쇼핑여가,20803.0,41.0,5.0,여의동
707447,1147055000.0,1156054000.0,20231015,10:00,10:00,1.0,1.0,차량,귀가,쇼핑여가,26692.0,33.0,6.0,여의동
707667,1162061500.0,1156054000.0,20231015,12:00,14:00,0.0,2.0,차량,귀가,쇼핑여가,35438.0,83.0,5.0,여의동
708929,1147055000.0,1156054000.0,20231015,12:00,12:00,1.0,1.0,차량,귀가,쇼핑여가,32412.0,27.0,6.0,여의동


In [25]:
df_s_date = df_s[df_s['date'] == 20231007]
df_s_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
292612,1156056000.0,1156054000.0,20231007,18:00,19:00,1.0,2.0,차량,귀가,기타,33425.0,58.0,9.0,여의동
295551,4141057000.0,1156054000.0,20231007,11:00,15:00,0.0,0.0,차량,기타,여행,82261.0,181.0,26.0,여의동
295772,1156067000.0,1156054000.0,20231007,16:00,17:00,1.0,0.0,차량,귀가,기타,17571.0,80.0,19.0,여의동
296215,1144058500.0,1156054000.0,20231007,16:00,17:00,1.0,0.0,차량,쇼핑여가,여행,9421.0,84.0,19.0,여의동
296923,1144059000.0,1156054000.0,20231007,19:00,19:00,1.0,3.0,차량,기타,기타,8923.0,17.0,9.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263625,1162066500.0,1156054000.0,20231007,11:00,12:00,1.0,3.0,차량,귀가,쇼핑여가,18333.0,52.0,5.0,여의동
263843,1156051500.0,1156054000.0,20231007,09:00,10:00,0.0,2.0,차량,귀가,기타,44967.0,41.0,5.0,여의동
265243,1154563000.0,1156054000.0,20231007,13:00,14:00,0.0,2.0,차량,귀가,쇼핑여가,36819.0,61.0,5.0,여의동
268425,1141058500.0,1156054000.0,20231007,16:00,17:00,0.0,2.0,차량,기타,기타,44891.0,80.0,5.0,여의동


In [113]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_s_date['dist_group'] = pd.cut(df_s_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_s_date['weight'] = df_s_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_s_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg = np.average(df_s_date['od_dist_avg'], weights=df_s_date['weight'])
print("가중 평균:", weighted_avg)

가중 평균: 66263.13916860313


<ipython-input-113-c996a805573e>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_s_date['dist_group'] = pd.cut(df_s_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-113-c996a805573e>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_s_date['weight'] = df_s_date['dist_group'].map(group_weights)
<ipython-input-113-c996a805573e>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The

In [114]:
# od_dist_avg 값들의 평균 계산
od_dist_avg_mean = weighted_avg
od_dist_avg_mean

66263.13916860313

In [115]:
#휘발유로 통일_탄소발자국 공식 사용

carbon_footprint_seoul = (od_dist_avg_mean / 16.04) * 2.097
carbon_footprint_seoul

8662.955289062393

시내버스 탄소발자국

In [28]:
df_sb = od_bus[od_bus['dest_hdong_cd'] == '1156054000.0']
df_sb

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
5032,1156051500.0,1156054000.0,20230901,13:00,13:00,1.0,2.0,시내버스,귀가,쇼핑여가,12739.0,19.0,7.0,여의동
321815,1159060500.0,1156054000.0,20230901,08:00,08:00,1.0,3.0,시내버스,귀가,기타,13771.0,26.0,5.0,여의동
332309,1144055500.0,1156054000.0,20230901,20:00,20:00,1.0,3.0,시내버스,귀가,기타,22459.0,29.0,5.0,여의동
355181,1144058500.0,1156054000.0,20230901,11:00,12:00,1.0,3.0,시내버스,업무,쇼핑여가,28239.0,33.0,5.0,여의동
358571,1144055500.0,1156054000.0,20230901,12:00,13:00,1.0,3.0,시내버스,여행,여행,12492.0,14.0,5.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672285,1156068000.0,1156054000.0,20231015,10:00,10:00,1.0,3.0,시내버스,귀가,쇼핑여가,8720.0,19.0,5.0,여의동
683870,1117063000.0,1156054000.0,20231015,13:00,14:00,1.0,3.0,시내버스,기타,쇼핑여가,40696.0,30.0,5.0,여의동
694156,1153051000.0,1156054000.0,20231015,11:00,11:00,0.0,3.0,시내버스,귀가,쇼핑여가,22942.0,32.0,5.0,여의동
698767,1141058500.0,1156054000.0,20231015,15:00,16:00,1.0,3.0,시내버스,쇼핑여가,쇼핑여가,64945.0,56.0,5.0,여의동


In [29]:
df_sb_date = df_sb[df_sb['date'] == 20231007]
df_sb_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
301510,1144068000.0,1156054000.0,20231007,17:00,18:00,0.0,2.0,시내버스,기타,기타,57564.0,45.0,7.0,여의동
308142,1153051000.0,1156054000.0,20231007,11:00,12:00,1.0,3.0,시내버스,귀가,쇼핑여가,14186.0,33.0,7.0,여의동
322822,4113551000.0,1156054000.0,20231007,09:00,10:00,1.0,0.0,시내버스,귀가,기타,29866.0,37.0,19.0,여의동
352949,1162052500.0,1156054000.0,20231007,11:00,12:00,0.0,0.0,시내버스,귀가,기타,6210.0,22.0,26.0,여의동
354736,1156069000.0,1156054000.0,20231007,19:00,19:00,0.0,1.0,시내버스,귀가,기타,16611.0,33.0,6.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217463,1156069000.0,1156054000.0,20231007,17:00,18:00,1.0,1.0,시내버스,귀가,쇼핑여가,25719.0,44.0,6.0,여의동
220633,1156056000.0,1156054000.0,20231007,16:00,17:00,0.0,2.0,시내버스,귀가,기타,54870.0,94.0,5.0,여의동
232043,1162064500.0,1156054000.0,20231007,17:00,18:00,1.0,1.0,시내버스,귀가,기타,32991.0,58.0,6.0,여의동
242128,1144059000.0,1156054000.0,20231007,17:00,18:00,0.0,3.0,시내버스,기타,기타,104904.0,56.0,5.0,여의동


In [118]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_sb_date['dist_group'] = pd.cut(df_sb_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_sb_date['weight'] = df_sb_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_sb_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_sb = np.average(df_sb_date['od_dist_avg'], weights=df_sb_date['weight'])
print("가중 평균:", weighted_avg_sb)

가중 평균: 33466.06826801517


<ipython-input-118-4ee08418c9b9>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sb_date['dist_group'] = pd.cut(df_sb_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-118-4ee08418c9b9>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sb_date['weight'] = df_sb_date['dist_group'].map(group_weights)
<ipython-input-118-4ee08418c9b9>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.

In [119]:
# od_dist_avg 값들의 평균 계산
od_dist_avg_mean_bus = weighted_avg_sb
od_dist_avg_mean_bus



33466.06826801517

In [120]:
 #보통 시내버스의 CNG 소비율은 km당 0.35 kg에서 0.5 kg 사이
 # 시내버스 경유로 통일

carbon_footprint_seoul = (od_dist_avg_mean_bus/ 15.35) * 2.582
carbon_footprint_seoul

5629.276108665484

지하철 탄소발자국

In [32]:
df_ss = od_subway[od_subway['dest_hdong_cd'] == '1156054000.0']
df_ss

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
1324,1114055000.0,1156054000.0,20230901,16:00,17:00,1.0,3.0,지하철,업무,쇼핑여가,28195.0,41.0,7.0,여의동
104928,1114052000.0,1156054000.0,20230901,18:00,18:00,0.0,3.0,지하철,업무,쇼핑여가,30122.0,38.0,7.0,여의동
110058,1144056500.0,1156054000.0,20230901,08:00,08:00,1.0,2.0,지하철,귀가,기타,22246.0,24.0,7.0,여의동
138846,1168064000.0,1156054000.0,20230901,18:00,19:00,1.0,3.0,지하철,업무,쇼핑여가,57873.0,65.0,9.0,여의동
154913,1150060300.0,1156054000.0,20230901,08:00,09:00,1.0,2.0,지하철,귀가,기타,21049.0,35.0,7.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
662699,1114058000.0,1156054000.0,20231015,11:00,12:00,0.0,2.0,지하철,여행,여행,18421.0,42.0,5.0,여의동
665391,1168064000.0,1156054000.0,20231015,17:00,18:00,0.0,2.0,지하철,쇼핑여가,쇼핑여가,22839.0,44.0,5.0,여의동
672282,1150053000.0,1156054000.0,20231015,11:00,12:00,0.0,2.0,지하철,귀가,쇼핑여가,22601.0,36.0,5.0,여의동
686534,1135069500.0,1156054000.0,20231015,10:00,12:00,0.0,3.0,지하철,기타,쇼핑여가,36423.0,90.0,5.0,여의동


In [33]:
df_ss_date = df_ss[df_ss['date'] == 20231007]
df_ss_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
299078,1126057000.0,1156054000.0,20231007,15:00,17:00,1.0,0.0,지하철,귀가,쇼핑여가,248360.0,79.0,19.0,여의동
302972,4157055000.0,1156054000.0,20231007,16:00,17:00,0.0,0.0,지하철,귀가,쇼핑여가,42312.0,76.0,26.0,여의동
303404,1171063100.0,1156054000.0,20231007,17:00,19:00,0.0,1.0,지하철,귀가,기타,208542.0,142.0,8.0,여의동
303529,1165053000.0,1156054000.0,20231007,17:00,19:00,1.0,0.0,지하철,쇼핑여가,여행,30843.0,104.0,19.0,여의동
305447,1129070500.0,1156054000.0,20231007,15:00,17:00,1.0,0.0,지하철,귀가,기타,69209.0,107.0,19.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250913,1150054000.0,1156054000.0,20231007,17:00,18:00,0.0,2.0,지하철,귀가,기타,58467.0,76.0,5.0,여의동
259617,1114062500.0,1156054000.0,20231007,17:00,18:00,0.0,2.0,지하철,귀가,기타,48062.0,81.0,5.0,여의동
259957,1120066000.0,1156054000.0,20231007,17:00,18:00,0.0,2.0,지하철,여행,여행,69979.0,79.0,5.0,여의동
261120,2826051500.0,1156054000.0,20231007,16:00,19:00,0.0,2.0,지하철,귀가,여행,122289.0,157.0,5.0,여의동


In [121]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_ss_date['dist_group'] = pd.cut(df_ss_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_ss_date['weight'] = df_ss_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_ss_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_ss= np.average(df_ss_date['od_dist_avg'], weights=df_ss_date['weight'])
weighted_avg_ss

<ipython-input-121-5d33ff79fbfd>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ss_date['dist_group'] = pd.cut(df_ss_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-121-5d33ff79fbfd>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ss_date['weight'] = df_ss_date['dist_group'].map(group_weights)
<ipython-input-121-5d33ff79fbfd>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.

65458.42894969108

In [122]:
# od_dist_avg 값들의 평균 계산
od_dist_avg_mean_subway = weighted_avg_ss
od_dist_avg_mean_subway



65458.42894969108

In [123]:
# 지하철은 1 km당 약 3~5 kWh의 전력을 소비
#(전기 사용량 * 0.4781)
carbon_footprint_seoul_subway = (od_dist_avg_mean_subway * 4 * 0.4781)/(30*24)
carbon_footprint_seoul_subway

173.8648604491517

철도

In [36]:
df_st = od_train[od_train['dest_hdong_cd'] == '1156054000.0']
df_st

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
437171,3611055600.0,1156054000.0,20230902,10:00,13:00,1.0,0.0,철도,귀가,기타,137722.0,196.0,19.0,여의동
141275,4141056000.0,1156054000.0,20230902,13:00,14:00,0.0,2.0,철도,귀가,기타,71724.0,55.0,5.0,여의동
334783,4136034000.0,1156054000.0,20230902,08:00,11:00,1.0,0.0,철도,귀가,여행,156998.0,144.0,19.0,여의동
565475,4111157300.0,1156054000.0,20230902,12:00,14:00,1.0,4.0,철도,귀가,쇼핑여가,178741.0,115.0,5.0,여의동
569607,4111356000.0,1156054000.0,20230902,11:00,12:00,1.0,3.0,철도,쇼핑여가,쇼핑여가,120289.0,50.0,5.0,여의동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
463088,4136051000.0,1156054000.0,20231013,18:00,20:00,0.0,0.0,철도,귀가,여행,57203.0,112.0,26.0,여의동
630036,3011067000.0,1156054000.0,20231014,14:00,16:00,0.0,0.0,철도,귀가,쇼핑여가,300079.0,150.0,26.0,여의동
694113,2726067000.0,1156054000.0,20231014,09:00,13:00,1.0,0.0,철도,귀가,여행,591645.0,224.0,19.0,여의동
543467,4413356000.0,1156054000.0,20231014,10:00,13:00,1.0,0.0,철도,귀가,쇼핑여가,178977.0,181.0,19.0,여의동


In [37]:
df_st_date = df_st[df_st['date'] == 20231007]
df_st_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
334677,4137057000.0,1156054000.0,20231007,10:00,12:00,1.0,1.0,철도,귀가,쇼핑여가,121907.0,96.0,6.0,여의동
379277,4131052000.0,1156054000.0,20231007,11:00,13:00,1.0,0.0,철도,귀가,여행,74978.0,116.0,19.0,여의동
433006,4122025300.0,1156054000.0,20231007,12:00,17:00,1.0,0.0,철도,귀가,기타,193289.0,291.0,19.0,여의동
437314,4182025000.0,1156054000.0,20231007,14:00,17:00,1.0,0.0,철도,귀가,기타,187561.0,218.0,19.0,여의동
476989,4159056000.0,1156054000.0,20231007,13:00,15:00,0.0,0.0,철도,귀가,기타,100748.0,156.0,26.0,여의동
493323,4127159000.0,1156054000.0,20231007,14:00,17:00,0.0,0.0,철도,귀가,기타,113399.0,145.0,26.0,여의동
509290,4122059000.0,1156054000.0,20231007,12:00,14:00,1.0,0.0,철도,기타,기타,119271.0,111.0,19.0,여의동
537629,4141062000.0,1156054000.0,20231007,17:00,18:00,0.0,0.0,철도,귀가,여행,62426.0,97.0,26.0,여의동
653597,4111356000.0,1156054000.0,20231007,12:00,13:00,0.0,2.0,철도,여행,여행,126013.0,82.0,5.0,여의동
705496,4141056000.0,1156054000.0,20231007,17:00,19:00,1.0,1.0,철도,귀가,여행,132717.0,139.0,6.0,여의동


In [124]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_st_date['dist_group'] = pd.cut(df_st_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_st_date['weight'] = df_st_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_st_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_st = np.average(df_st_date['od_dist_avg'], weights=df_st_date['weight'])
print("가중 평균:", weighted_avg_st)

가중 평균: 156971.65697674418


<ipython-input-124-4496dc083db9>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_st_date['dist_group'] = pd.cut(df_st_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-124-4496dc083db9>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_st_date['weight'] = df_st_date['dist_group'].map(group_weights)
<ipython-input-124-4496dc083db9>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.

In [125]:
# od_dist_avg 값들의 평균 계산
od_dist_avg_mean_train = weighted_avg_st
od_dist_avg_mean_train

156971.65697674418

In [126]:
# KTX는 주행 시 평균적으로 1km당 약 2.5 kWh에서 3 kWh의 전력 소비
#(전기 사용량 * 0.4781)
carbon_footprint_seoul_train = (od_dist_avg_mean_train * 2.5 * 0.4781) / (200*24)
carbon_footprint_seoul_train

39.08757770863615

# 임실

In [40]:
df_I = od_car[od_car['dest_hdong_cd'] == '4575034000.0']
df_I

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
433555,4511366500.0,4575034000.0,20230901,11:00,12:00,0.0,3.0,차량,기타,기타,72899.0,43.0,5.0,임실 성수면
500719,4575025000.0,4575034000.0,20230901,13:00,13:00,1.0,4.0,차량,기타,기타,4452.0,16.0,6.0,임실 성수면
533970,4511364100.0,4575034000.0,20230901,17:00,19:00,0.0,3.0,차량,귀가,기타,119755.0,95.0,5.0,임실 성수면
646223,4575025000.0,4575034000.0,20230901,08:00,08:00,0.0,3.0,차량,귀가,기타,9456.0,32.0,5.0,임실 성수면
163041,4575025000.0,4575034000.0,20230901,11:00,11:00,1.0,6.0,차량,기타,기타,9061.0,13.0,7.0,임실 성수면
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595523,2914082100.0,4575034000.0,20231015,13:00,15:00,0.0,3.0,차량,귀가,여행,286538.0,168.0,5.0,임실 성수면
615468,4511354000.0,4575034000.0,20231015,09:00,11:00,0.0,3.0,차량,여행,여행,108036.0,111.0,5.0,임실 성수면
629618,4511173000.0,4575034000.0,20231015,10:00,11:00,0.0,3.0,차량,귀가,기타,114664.0,112.0,5.0,임실 성수면
640075,4511173000.0,4575034000.0,20231015,08:00,09:00,0.0,3.0,차량,여행,여행,61838.0,41.0,5.0,임실 성수면


In [41]:
df_I_date=df_I[(df_I['date']>= 20231006) & (df_I['date'] <= 20231009)]
df_I_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
175117,4575025000.0,4575034000.0,20231006,14:00,15:00,0.0,4.0,차량,여행,여행,35845.0,42.0,8.0,임실 성수면
182977,4575025000.0,4575034000.0,20231006,13:00,14:00,1.0,6.0,차량,여행,여행,32899.0,42.0,7.0,임실 성수면
208629,4575025000.0,4575034000.0,20231006,15:00,15:00,1.0,7.0,차량,기타,기타,3750.0,16.0,13.0,임실 성수면
208694,4575025000.0,4575034000.0,20231006,16:00,16:00,0.0,5.0,차량,기타,기타,14991.0,16.0,9.0,임실 성수면
227667,4575032000.0,4575034000.0,20231006,16:00,17:00,0.0,3.0,차량,기타,기타,63950.0,50.0,9.0,임실 성수면
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173108,4519056000.0,4575034000.0,20231009,14:00,15:00,0.0,3.0,차량,귀가,기타,57767.0,39.0,5.0,임실 성수면
175106,2917067700.0,4575034000.0,20231009,11:00,14:00,0.0,3.0,차량,기타,기타,187107.0,153.0,5.0,임실 성수면
179463,4575025000.0,4575034000.0,20231009,15:00,15:00,0.0,6.0,차량,여행,여행,2138.0,15.0,5.0,임실 성수면
180362,4513055000.0,4575034000.0,20231009,11:00,14:00,0.0,3.0,차량,귀가,기타,283414.0,199.0,5.0,임실 성수면


In [127]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_I_date['dist_group'] = pd.cut(df_I_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_I_date['weight'] = df_I_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_I_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_I = np.average(df_I_date['od_dist_avg'], weights=df_I_date['weight'])
print("가중 평균:", weighted_avg_I)

가중 평균: 154565.79491193738


<ipython-input-127-817e3e1e9c7b>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_I_date['dist_group'] = pd.cut(df_I_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-127-817e3e1e9c7b>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_I_date['weight'] = df_I_date['dist_group'].map(group_weights)
<ipython-input-127-817e3e1e9c7b>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The

In [129]:
# od_dist_avg 값들의 평균 계산
dist_I = weighted_avg_I
dist_I

154565.79491193738

In [130]:
#휘발유로 통일_탄소발자국 공식 사용

carbon_footprint_imsil = (dist_I / 16.04) * 2.097
carbon_footprint_imsil

20207.26134229007

시내버스

In [44]:
df_I_b = od_bus[od_bus['dest_hdong_cd'] == '4575034000.0']
df_I_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
463291,4575025000.0,4575034000.0,20230901,08:00,09:00,0.0,7.0,시내버스,귀가,기타,30384.0,54.0,5.0,임실 성수면
556934,4575035500.0,4575034000.0,20230901,16:00,16:00,0.0,3.0,시내버스,여행,여행,40892.0,26.0,5.0,임실 성수면
108930,4575025000.0,4575034000.0,20230901,08:00,08:00,0.0,3.0,시내버스,귀가,기타,20239.0,17.0,5.0,임실 성수면
527626,4575025000.0,4575034000.0,20230902,10:00,11:00,0.0,3.0,시내버스,귀가,기타,30072.0,29.0,5.0,임실 성수면
606582,4575031000.0,4575034000.0,20230902,15:00,16:00,0.0,3.0,시내버스,기타,기타,61956.0,51.0,5.0,임실 성수면
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
646582,4575025000.0,4575034000.0,20231015,11:00,11:00,0.0,3.0,시내버스,여행,여행,7741.0,16.0,5.0,임실 성수면
683918,4575038000.0,4575034000.0,20231015,16:00,16:00,0.0,3.0,시내버스,쇼핑여가,기타,27296.0,23.0,5.0,임실 성수면
680758,4518043000.0,4575034000.0,20231015,11:00,12:00,0.0,3.0,시내버스,기타,기타,44886.0,45.0,5.0,임실 성수면
59649,4575032000.0,4575034000.0,20231015,13:00,14:00,0.0,3.0,시내버스,기타,기타,128670.0,64.0,5.0,임실 성수면


In [45]:
df_I_date_b=df_I_b[(df_I_b['date']>= 20231006) & (df_I_b['date'] <= 20231009)]
df_I_date_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
519622,4575025000.0,4575034000.0,20231006,11:00,11:00,0.0,5.0,시내버스,기타,기타,22145.0,20.0,6.0,임실 성수면
620274,4575039000.0,4575034000.0,20231006,15:00,16:00,0.0,5.0,시내버스,귀가,기타,45698.0,47.0,6.0,임실 성수면
70246,4575025000.0,4575034000.0,20231006,14:00,15:00,0.0,2.0,시내버스,기타,기타,68761.0,59.0,5.0,임실 성수면
89766,4575025000.0,4575034000.0,20231006,13:00,14:00,0.0,6.0,시내버스,기타,기타,20278.0,32.0,5.0,임실 성수면
460840,4575025000.0,4575034000.0,20231006,11:00,11:00,0.0,2.0,시내버스,귀가,기타,52809.0,20.0,7.0,임실 성수면
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159408,4575025000.0,4575034000.0,20231009,11:00,11:00,0.0,3.0,시내버스,귀가,기타,61636.0,45.0,5.0,임실 성수면
292271,4575039000.0,4575034000.0,20231009,11:00,13:00,0.0,3.0,시내버스,기타,기타,142684.0,106.0,5.0,임실 성수면
413188,4575038000.0,4575034000.0,20231009,11:00,11:00,0.0,3.0,시내버스,기타,기타,40208.0,19.0,9.0,임실 성수면
121543,4571032000.0,4575034000.0,20231009,09:00,11:00,0.0,3.0,시내버스,여행,여행,87468.0,90.0,5.0,임실 성수면


In [132]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_I_date_b['dist_group'] = pd.cut(df_I_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_I_date_b['weight'] = df_I_date_b['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_I_date_b['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_bI = np.average(df_I_date_b['od_dist_avg'], weights=df_I_date_b['weight'])
print("가중 평균:", weighted_avg_bI)

가중 평균: 87427.89655172414


<ipython-input-132-2aef1ab166de>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_I_date_b['dist_group'] = pd.cut(df_I_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-132-2aef1ab166de>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_I_date_b['weight'] = df_I_date_b['dist_group'].map(group_weights)
<ipython-input-132-2aef1ab166de>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [133]:
# od_dist_avg 값들의 평균 계산
od_dist_avg_mean_bus = weighted_avg_bI
od_dist_avg_mean_bus


87427.89655172414

In [134]:
 #보통 시내버스의 CNG 소비율은 km당 0.35 kg에서 0.5 kg 사이
 # 시내버스는 CNG(천연가스)로 통일. 주행 거리 (km)×0.4kg/km×2.75kg CO2/kg   -> 공식 이상한듯
 # 시내버스 경유로 통일

carbon_footprint_imsil = (od_dist_avg_mean_bus/ 15.35) * 2.582
carbon_footprint_imsil

14706.112631697182

In [48]:
df_I_s = od_subway[od_subway['dest_hdong_cd'] == '4575034000.0']
df_I_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination


In [49]:
df_I_t = od_train[od_train['dest_hdong_cd'] == '4575034000.0']
df_I_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
714340,4511361100.0,4575034000.0,20231002,11:00,12:00,0.0,3.0,철도,여행,여행,67983.0,31.0,5.0,임실 성수면
240249,4511364200.0,4575034000.0,20231007,10:00,12:00,0.0,3.0,철도,여행,여행,166858.0,110.0,5.0,임실 성수면
269039,4511364200.0,4575034000.0,20231009,11:00,12:00,0.0,3.0,철도,기타,기타,99919.0,53.0,5.0,임실 성수면


In [50]:
df_I_date_t=df_I_t[(df_I_t['date']>= 20231006) & (df_I_t['date'] <= 20231009)]
df_I_date_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
240249,4511364200.0,4575034000.0,20231007,10:00,12:00,0.0,3.0,철도,여행,여행,166858.0,110.0,5.0,임실 성수면
269039,4511364200.0,4575034000.0,20231009,11:00,12:00,0.0,3.0,철도,기타,기타,99919.0,53.0,5.0,임실 성수면


In [135]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_I_date_t['dist_group'] = pd.cut(df_I_date_t['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_I_date_t['weight'] = df_I_date_t['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_I_date_t['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_tI = np.average(df_I_date_t['od_dist_avg'], weights=df_I_date_t['weight'])
print("가중 평균:", weighted_avg_tI)

가중 평균: 140082.40000000002


<ipython-input-135-c01167fc88a1>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_I_date_t['dist_group'] = pd.cut(df_I_date_t['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-135-c01167fc88a1>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_I_date_t['weight'] = df_I_date_t['dist_group'].map(group_weights)
<ipython-input-135-c01167fc88a1>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [136]:
# od_dist_avg 값들의 평균 계산
od_dist_avg_mean_I_t = weighted_avg_tI
od_dist_avg_mean_I_t


140082.40000000002

In [137]:
# KTX는 주행 시 평균적으로 1km당 약 2.5 kWh에서 3 kWh의 전력 소비
#(전기 사용량 * 0.4781)
carbon_footprint_imsil_train = (od_dist_avg_mean_I_t * 2.5 * 0.4781) / (200*24)
carbon_footprint_imsil_train

34.88197679166667

# 강릉(스피드스케이트장)

In [53]:
df_GA = od_car[od_car['dest_hdong_cd'] == '5115057200.0']
df_GA

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
9752,5115059000.0,5115057200.0,20230901,16:00,17:00,1.0,5.0,차량,여행,여행,16581.0,32.0,9.0,강릉시 포남2동
25339,5115036000.0,5115057200.0,20230901,18:00,19:00,0.0,2.0,차량,귀가,쇼핑여가,31848.0,28.0,11.0,강릉시 포남2동
49688,5115031000.0,5115057200.0,20230901,16:00,18:00,1.0,5.0,차량,여행,여행,123083.0,84.0,9.0,강릉시 포남2동
65124,5115064500.0,5115057200.0,20230901,19:00,19:00,1.0,2.0,차량,귀가,기타,23158.0,36.0,8.0,강릉시 포남2동
75640,1153053000.0,5115057200.0,20230901,15:00,19:00,0.0,0.0,차량,귀가,여행,206895.0,238.0,35.0,강릉시 포남2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710615,4128157700.0,5115057200.0,20231015,11:00,15:00,1.0,3.0,차량,귀가,여행,539169.0,211.0,5.0,강릉시 포남2동
711120,5115052000.0,5115057200.0,20231015,12:00,13:00,1.0,4.0,차량,쇼핑여가,쇼핑여가,40098.0,14.0,5.0,강릉시 포남2동
713687,5115057100.0,5115057200.0,20231015,20:00,20:00,0.0,2.0,차량,쇼핑여가,쇼핑여가,7792.0,16.0,5.0,강릉시 포남2동
714799,5115066500.0,5115057200.0,20231015,13:00,15:00,1.0,1.0,차량,쇼핑여가,여행,320011.0,101.0,6.0,강릉시 포남2동


In [54]:
df_ga_date=df_GA[(df_GA['date']>= 20231012) & (df_GA['date'] <= 20231015)]
df_ga_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
239542,5115054000.0,5115057200.0,20231012,19:00,20:00,1.0,0.0,차량,쇼핑여가,기타,37448.0,45.0,26.0,강릉시 포남2동
248750,5115060000.0,5115057200.0,20231012,14:00,14:00,1.0,5.0,차량,기타,기타,27949.0,28.0,9.0,강릉시 포남2동
254307,5115059000.0,5115057200.0,20231012,15:00,15:00,0.0,5.0,차량,기타,기타,5425.0,8.0,14.0,강릉시 포남2동
276111,5115066500.0,5115057200.0,20231012,15:00,15:00,0.0,3.0,차량,여행,여행,5026.0,11.0,13.0,강릉시 포남2동
283940,5115055000.0,5115057200.0,20231012,10:00,10:00,1.0,4.0,차량,귀가,기타,15064.0,34.0,10.0,강릉시 포남2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710615,4128157700.0,5115057200.0,20231015,11:00,15:00,1.0,3.0,차량,귀가,여행,539169.0,211.0,5.0,강릉시 포남2동
711120,5115052000.0,5115057200.0,20231015,12:00,13:00,1.0,4.0,차량,쇼핑여가,쇼핑여가,40098.0,14.0,5.0,강릉시 포남2동
713687,5115057100.0,5115057200.0,20231015,20:00,20:00,0.0,2.0,차량,쇼핑여가,쇼핑여가,7792.0,16.0,5.0,강릉시 포남2동
714799,5115066500.0,5115057200.0,20231015,13:00,15:00,1.0,1.0,차량,쇼핑여가,여행,320011.0,101.0,6.0,강릉시 포남2동


In [138]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_ga_date['dist_group'] = pd.cut(df_ga_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_ga_date['weight'] = df_ga_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_ga_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_ga = np.average(df_ga_date['od_dist_avg'], weights=df_ga_date['weight'])
print("가중 평균:", weighted_avg_ga)

가중 평균: 119401.0585635359


<ipython-input-138-78f0a9bc7ee3>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date['dist_group'] = pd.cut(df_ga_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-138-78f0a9bc7ee3>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date['weight'] = df_ga_date['dist_group'].map(group_weights)
<ipython-input-138-78f0a9bc7ee3>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.

In [139]:
# od_dist_avg 값들의 평균 계산
dist_ga = weighted_avg_ga
dist_ga

119401.0585635359

In [140]:
#휘발유로 통일_탄소발자국 공식 사용

carbon_footprint_ga = (dist_ga/ 16.04) * 2.097
carbon_footprint_ga

15609.976297240324

In [57]:
df_GA_b = od_bus[od_bus['dest_hdong_cd'] == '5115057200.0']
df_GA_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
27874,5115059000.0,5115057200.0,20230901,08:00,08:00,0.0,1.0,시내버스,귀가,기타,21231.0,16.0,65.0,강릉시 포남2동
79353,5115064500.0,5115057200.0,20230901,18:00,18:00,1.0,1.0,시내버스,귀가,쇼핑여가,15985.0,18.0,11.0,강릉시 포남2동
151010,5115059000.0,5115057200.0,20230901,08:00,08:00,0.0,3.0,시내버스,귀가,기타,10581.0,10.0,8.0,강릉시 포남2동
217855,5115055000.0,5115057200.0,20230901,14:00,15:00,0.0,0.0,시내버스,기타,쇼핑여가,73433.0,76.0,35.0,강릉시 포남2동
273614,5115059000.0,5115057200.0,20230901,14:00,14:00,0.0,0.0,시내버스,귀가,기타,25457.0,28.0,35.0,강릉시 포남2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676141,5115066500.0,5115057200.0,20231015,14:00,14:00,1.0,3.0,시내버스,여행,여행,11356.0,12.0,5.0,강릉시 포남2동
681668,5115066500.0,5115057200.0,20231015,18:00,18:00,1.0,4.0,시내버스,여행,쇼핑여가,20833.0,18.0,5.0,강릉시 포남2동
683105,5115064500.0,5115057200.0,20231015,18:00,18:00,0.0,2.0,시내버스,기타,기타,14732.0,16.0,5.0,강릉시 포남2동
689586,5115055000.0,5115057200.0,20231015,11:00,11:00,1.0,3.0,시내버스,귀가,기타,6273.0,7.0,5.0,강릉시 포남2동


In [58]:
df_ga_date_b=df_GA_b[(df_GA_b['date']>= 20231012) & (df_GA_b['date'] <= 20231015)]
df_ga_date_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
378888,5115051000.0,5115057200.0,20231012,13:00,13:00,1.0,3.0,시내버스,귀가,기타,17343.0,15.0,10.0,강릉시 포남2동
464387,5115051000.0,5115057200.0,20231012,14:00,15:00,1.0,0.0,시내버스,귀가,기타,11390.0,50.0,26.0,강릉시 포남2동
527900,5115064500.0,5115057200.0,20231012,14:00,14:00,1.0,1.0,시내버스,여행,여행,15115.0,12.0,11.0,강릉시 포남2동
556169,5115051000.0,5115057200.0,20231012,12:00,12:00,1.0,4.0,시내버스,귀가,기타,13474.0,18.0,5.0,강릉시 포남2동
571172,5115056000.0,5115057200.0,20231012,16:00,17:00,1.0,1.0,시내버스,기타,쇼핑여가,32277.0,79.0,6.0,강릉시 포남2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676141,5115066500.0,5115057200.0,20231015,14:00,14:00,1.0,3.0,시내버스,여행,여행,11356.0,12.0,5.0,강릉시 포남2동
681668,5115066500.0,5115057200.0,20231015,18:00,18:00,1.0,4.0,시내버스,여행,쇼핑여가,20833.0,18.0,5.0,강릉시 포남2동
683105,5115064500.0,5115057200.0,20231015,18:00,18:00,0.0,2.0,시내버스,기타,기타,14732.0,16.0,5.0,강릉시 포남2동
689586,5115055000.0,5115057200.0,20231015,11:00,11:00,1.0,3.0,시내버스,귀가,기타,6273.0,7.0,5.0,강릉시 포남2동


In [141]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_ga_date_b['dist_group'] = pd.cut(df_ga_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_ga_date_b['weight'] = df_ga_date_b['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_ga_date_b['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_gab = np.average(df_ga_date_b['od_dist_avg'], weights=df_ga_date_b['weight'])
print("가중 평균:", weighted_avg_gab)

가중 평균: 45112.93803786575


<ipython-input-141-a1a03bc4f533>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date_b['dist_group'] = pd.cut(df_ga_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-141-a1a03bc4f533>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date_b['weight'] = df_ga_date_b['dist_group'].map(group_weights)
<ipython-input-141-a1a03bc4f533>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace

In [142]:
dist_ga_bus = weighted_avg_gab
dist_ga_bus

45112.93803786575

In [143]:
 #보통 시내버스의 CNG 소비율은 km당 0.35 kg에서 0.5 kg 사이
 # 시내버스는 CNG(천연가스)로 통일. 주행 거리 (km)×0.4kg/km×2.75kg CO2/kg   -> 공식 이상한듯
 # 시내버스 경유로 통일

carbon_footprint_bus = (dist_ga_bus/ 15.35) * 2.582
carbon_footprint_bus

7588.378241939373

지하철

In [61]:
df_GA_s= od_subway[od_subway['dest_hdong_cd'] == '5115057200.0']
df_GA_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
476977,4128157600.0,5115057200.0,20230928,18:00,22:00,1.0,4.0,지하철,귀가,여행,296260.0,260.0,5.0,강릉시 포남2동
312240,2811062800.0,5115057200.0,20231003,17:00,22:00,1.0,0.0,지하철,<NA>,기타,729133.0,305.0,26.0,강릉시 포남2동
84461,1159060500.0,5115057200.0,20231013,13:00,17:00,1.0,1.0,지하철,여행,여행,301070.0,247.0,6.0,강릉시 포남2동


In [62]:
df_ga_date_s=df_GA_s[(df_GA_s['date']>= 20231012) & (df_GA_s['date'] <= 20231015)]
df_ga_date_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
84461,1159060500.0,5115057200.0,20231013,13:00,17:00,1.0,1.0,지하철,여행,여행,301070.0,247.0,6.0,강릉시 포남2동


In [144]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_ga_date_s['dist_group'] = pd.cut(df_ga_date_s['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_ga_date_s['weight'] = df_ga_date_s['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_ga_date_s['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_gas = np.average(df_ga_date_s['od_dist_avg'], weights=df_ga_date_s['weight'])
print("가중 평균:", weighted_avg_gas)

가중 평균: 301070.0


<ipython-input-144-70319d997b5a>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date_s['dist_group'] = pd.cut(df_ga_date_s['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-144-70319d997b5a>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date_s['weight'] = df_ga_date_s['dist_group'].map(group_weights)
<ipython-input-144-70319d997b5a>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace

In [145]:
dist_ga_subway = weighted_avg_gas
dist_ga_subway

301070.0

In [146]:
# 지하철은 1 km당 약 3~5 kWh의 전력을 소비
#(전기 사용량 * 0.4781)
carbon_footprint_ga_subway = (dist_ga_subway * 4 * 0.4781) / (30*24)
carbon_footprint_ga_subway

799.6753722222222

철도

In [65]:
df_GA_t= od_train[od_train['dest_hdong_cd'] == '5115057200.0']
df_GA_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
655811,1123057000.0,5115057200.0,20230901,11:00,14:00,0.0,2.0,철도,여행,여행,420543.0,150.0,5.0,강릉시 포남2동
16847,1123056000.0,5115057200.0,20230901,11:00,14:00,1.0,3.0,철도,기타,여행,387288.0,132.0,5.0,강릉시 포남2동
500009,1123056000.0,5115057200.0,20230901,21:00,23:00,1.0,3.0,철도,여행,여행,391050.0,123.0,5.0,강릉시 포남2동
580952,1123070500.0,5115057200.0,20230901,18:00,20:00,0.0,2.0,철도,기타,여행,396603.0,139.0,5.0,강릉시 포남2동
49677,1117053000.0,5115057200.0,20230901,09:00,12:00,1.0,4.0,철도,여행,여행,407603.0,190.0,5.0,강릉시 포남2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327343,1168051000.0,5115057200.0,20231014,09:00,13:00,1.0,3.0,철도,귀가,여행,400465.0,228.0,5.0,강릉시 포남2동
302381,1117055500.0,5115057200.0,20231014,15:00,19:00,1.0,3.0,철도,기타,기타,489849.0,213.0,5.0,강릉시 포남2동
714560,1129058000.0,5115057200.0,20231015,19:00,22:00,1.0,3.0,철도,귀가,여행,427670.0,182.0,5.0,강릉시 포남2동
60994,1165058100.0,5115057200.0,20231015,18:00,22:00,1.0,4.0,철도,기타,기타,744523.0,200.0,5.0,강릉시 포남2동


In [66]:
df_ga_date_t=df_GA_t[(df_GA_t['date']>= 20231012) & (df_GA_t['date'] <= 20231015)]
df_ga_date_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
123768,5173025000.0,5115057200.0,20231013,19:00,21:00,1.0,3.0,철도,기타,기타,143945.0,104.0,5.0,강릉시 포남2동
162360,1117062500.0,5115057200.0,20231013,11:00,15:00,1.0,4.0,철도,귀가,여행,413208.0,248.0,5.0,강릉시 포남2동
661951,5113032000.0,5115057200.0,20231013,08:00,10:00,1.0,4.0,철도,기타,기타,179458.0,126.0,5.0,강릉시 포남2동
67379,1165052000.0,5115057200.0,20231013,18:00,22:00,1.0,3.0,철도,기타,여행,346638.0,190.0,5.0,강릉시 포남2동
128602,1123056000.0,5115057200.0,20231013,12:00,14:00,0.0,2.0,철도,여행,여행,395279.0,118.0,5.0,강릉시 포남2동
213683,4113562000.0,5115057200.0,20231013,13:00,16:00,1.0,3.0,철도,여행,여행,407470.0,223.0,5.0,강릉시 포남2동
688514,1117053000.0,5115057200.0,20231013,12:00,15:00,1.0,3.0,철도,쇼핑여가,쇼핑여가,636125.0,192.0,5.0,강릉시 포남2동
32997,1123056000.0,5115057200.0,20231014,12:00,14:00,0.0,2.0,철도,여행,여행,402994.0,129.0,5.0,강릉시 포남2동
495518,1117069000.0,5115057200.0,20231014,10:00,12:00,1.0,4.0,철도,기타,쇼핑여가,520200.0,156.0,10.0,강릉시 포남2동
133174,1123056000.0,5115057200.0,20231014,12:00,14:00,0.0,2.0,철도,기타,여행,665470.0,140.0,5.0,강릉시 포남2동


In [147]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_ga_date_t['dist_group'] = pd.cut(df_ga_date_t['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_ga_date_t['weight'] = df_ga_date_t['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_ga_date_t['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_gat = np.average(df_ga_date_t['od_dist_avg'], weights=df_ga_date_t['weight'])
print("가중 평균:", weighted_avg_gat)

가중 평균: 455939.01449275366


<ipython-input-147-52d6ecf93f9f>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date_t['dist_group'] = pd.cut(df_ga_date_t['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-147-52d6ecf93f9f>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ga_date_t['weight'] = df_ga_date_t['dist_group'].map(group_weights)
<ipython-input-147-52d6ecf93f9f>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace

In [148]:
dist_ga_train = weighted_avg_gat
dist_ga_train

455939.01449275366

In [149]:
# KTX는 주행 시 평균적으로 1km당 약 2.5 kWh에서 3 kWh의 전력 소비
#(전기 사용량 * 0.4781)
carbon_footprint_seoul_train = (dist_ga_train * 2.5 * 0.4781) / (200*24)
carbon_footprint_seoul_train

113.53356397342996

# 강릉(경포호수공원)

In [69]:
df_GB = od_car[od_car['dest_hdong_cd'] == '5115058000.0' ]
df_GB

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
9441,5115066500.0,5115058000.0,20230901,17:00,17:00,1.0,2.0,차량,여행,여행,6008.0,7.0,16.0,강릉시 초당동
17345,5115066500.0,5115058000.0,20230901,16:00,16:00,0.0,4.0,차량,여행,여행,24967.0,27.0,8.0,강릉시 초당동
26241,5115066500.0,5115058000.0,20230901,11:00,11:00,1.0,5.0,차량,여행,여행,20506.0,20.0,9.0,강릉시 초당동
27330,5115066500.0,5115058000.0,20230901,08:00,08:00,0.0,3.0,차량,여행,여행,7392.0,15.0,10.0,강릉시 초당동
28147,5115066500.0,5115058000.0,20230901,17:00,17:00,0.0,4.0,차량,여행,여행,6477.0,13.0,8.0,강릉시 초당동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
630175,5176034000.0,5115058000.0,20231015,12:00,14:00,0.0,4.0,차량,여행,여행,227797.0,94.0,6.0,강릉시 초당동
635330,5115064500.0,5115058000.0,20231015,11:00,12:00,0.0,5.0,차량,귀가,기타,14623.0,22.0,5.0,강릉시 초당동
640355,5176038000.0,5115058000.0,20231015,13:00,14:00,0.0,3.0,차량,여행,여행,92098.0,45.0,5.0,강릉시 초당동
663406,5115051000.0,5115058000.0,20231015,11:00,11:00,0.0,3.0,차량,여행,여행,29664.0,19.0,5.0,강릉시 초당동


In [70]:
df_gb_date=df_GB[(df_GB['date']>= 20231012) & (df_GB['date'] <= 20231015)]
df_gb_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
234227,5115055000.0,5115058000.0,20231012,12:00,13:00,0.0,1.0,차량,여행,여행,37976.0,51.0,7.0,강릉시 초당동
243944,5115066500.0,5115058000.0,20231012,18:00,18:00,1.0,2.0,차량,여행,여행,16241.0,14.0,11.0,강릉시 초당동
247857,5115066500.0,5115058000.0,20231012,21:00,21:00,0.0,1.0,차량,여행,여행,1037.0,4.0,7.0,강릉시 초당동
262733,5115066500.0,5115058000.0,20231012,10:00,11:00,0.0,4.0,차량,여행,여행,34741.0,46.0,8.0,강릉시 초당동
267515,5115066500.0,5115058000.0,20231012,12:00,12:00,1.0,3.0,차량,여행,여행,13181.0,13.0,10.0,강릉시 초당동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
630175,5176034000.0,5115058000.0,20231015,12:00,14:00,0.0,4.0,차량,여행,여행,227797.0,94.0,6.0,강릉시 초당동
635330,5115064500.0,5115058000.0,20231015,11:00,12:00,0.0,5.0,차량,귀가,기타,14623.0,22.0,5.0,강릉시 초당동
640355,5176038000.0,5115058000.0,20231015,13:00,14:00,0.0,3.0,차량,여행,여행,92098.0,45.0,5.0,강릉시 초당동
663406,5115051000.0,5115058000.0,20231015,11:00,11:00,0.0,3.0,차량,여행,여행,29664.0,19.0,5.0,강릉시 초당동


In [150]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_gb_date['dist_group'] = pd.cut(df_gb_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_gb_date['weight'] = df_gb_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_gb_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_gb = np.average(df_gb_date['od_dist_avg'], weights=df_gb_date['weight'])
print("가중 평균:", weighted_avg_gb)

가중 평균: 88512.6693108577


<ipython-input-150-292dee64ed03>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gb_date['dist_group'] = pd.cut(df_gb_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-150-292dee64ed03>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gb_date['weight'] = df_gb_date['dist_group'].map(group_weights)
<ipython-input-150-292dee64ed03>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.

In [151]:
# od_dist_avg 값들의 평균 계산
dist_gb = weighted_avg_gb
dist_gb

88512.6693108577

In [152]:
#휘발유로 통일_탄소발자국 공식 사용

carbon_footprint_gb = (dist_gb / 16.04) * 2.097
carbon_footprint_gb

11571.762315764876

시내버스


In [73]:
df_GB_b= od_bus[od_bus['dest_hdong_cd'] == '5115058000.0' ]
df_GB_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
24445,5115066500.0,5115058000.0,20230901,10:00,10:00,0.0,4.0,시내버스,여행,여행,16646.0,11.0,8.0,강릉시 초당동
89704,5115059000.0,5115058000.0,20230901,12:00,12:00,1.0,3.0,시내버스,여행,여행,22088.0,21.0,7.0,강릉시 초당동
184465,5115057200.0,5115058000.0,20230901,14:00,15:00,0.0,1.0,시내버스,귀가,쇼핑여가,17859.0,7.0,7.0,강릉시 초당동
237536,5115056000.0,5115058000.0,20230901,13:00,13:00,0.0,1.0,시내버스,여행,여행,44885.0,17.0,7.0,강릉시 초당동
261595,5115054000.0,5115058000.0,20230901,08:00,08:00,0.0,1.0,시내버스,귀가,기타,4927.0,18.0,7.0,강릉시 초당동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426713,5115056000.0,5115058000.0,20231015,14:00,15:00,0.0,5.0,시내버스,귀가,기타,24033.0,39.0,5.0,강릉시 초당동
483479,5115066500.0,5115058000.0,20231015,10:00,11:00,1.0,5.0,시내버스,여행,여행,34063.0,32.0,6.0,강릉시 초당동
550105,5115064500.0,5115058000.0,20231015,11:00,12:00,0.0,3.0,시내버스,기타,기타,13915.0,17.0,5.0,강릉시 초당동
613857,5115066500.0,5115058000.0,20231015,13:00,14:00,0.0,5.0,시내버스,여행,여행,12883.0,14.0,5.0,강릉시 초당동


In [74]:
df_gb_date_b=df_GB_b[(df_GB_b['date']>= 20231012) & (df_GB_b['date'] <= 20231015)]
df_gb_date_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
363010,5115066500.0,5115058000.0,20231012,16:00,16:00,1.0,3.0,시내버스,여행,여행,17598.0,8.0,7.0,강릉시 초당동
417009,5115054000.0,5115058000.0,20231012,13:00,14:00,0.0,1.0,시내버스,쇼핑여가,여행,10897.0,14.0,7.0,강릉시 초당동
488678,5115036000.0,5115058000.0,20231012,11:00,14:00,0.0,0.0,시내버스,여행,여행,105141.0,148.0,24.0,강릉시 초당동
659283,5115066500.0,5115058000.0,20231012,17:00,18:00,0.0,4.0,시내버스,기타,기타,44666.0,35.0,6.0,강릉시 초당동
81551,5115064500.0,5115058000.0,20231012,14:00,14:00,1.0,2.0,시내버스,여행,여행,9385.0,7.0,5.0,강릉시 초당동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426713,5115056000.0,5115058000.0,20231015,14:00,15:00,0.0,5.0,시내버스,귀가,기타,24033.0,39.0,5.0,강릉시 초당동
483479,5115066500.0,5115058000.0,20231015,10:00,11:00,1.0,5.0,시내버스,여행,여행,34063.0,32.0,6.0,강릉시 초당동
550105,5115064500.0,5115058000.0,20231015,11:00,12:00,0.0,3.0,시내버스,기타,기타,13915.0,17.0,5.0,강릉시 초당동
613857,5115066500.0,5115058000.0,20231015,13:00,14:00,0.0,5.0,시내버스,여행,여행,12883.0,14.0,5.0,강릉시 초당동


In [153]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_gb_date_b['dist_group'] = pd.cut(df_gb_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_gb_date_b['weight'] = df_gb_date_b['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_gb_date_b['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_gbb = np.average(df_gb_date_b['od_dist_avg'], weights=df_gb_date_b['weight'])
print("가중 평균:", weighted_avg_gbb)

가중 평균: 33268.33933933935


<ipython-input-153-a9472f9c47e6>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gb_date_b['dist_group'] = pd.cut(df_gb_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-153-a9472f9c47e6>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gb_date_b['weight'] = df_gb_date_b['dist_group'].map(group_weights)
<ipython-input-153-a9472f9c47e6>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace

In [154]:
# od_dist_avg 값들의 평균 계산
dist_gb_b = weighted_avg_gbb
dist_gb_b

33268.33933933935

In [155]:
 #보통 시내버스의 CNG 소비율은 km당 0.35 kg에서 0.5 kg 사이
 # 시내버스는 CNG(천연가스)로 통일. 주행 거리 (km)×0.4kg/km×2.75kg CO2/kg   -> 공식 이상한듯
 # 시내버스 경유로 통일

carbon_footprint_gb_bus = (dist_gb_b/ 15.35) * 2.582
carbon_footprint_gb_bus

5596.016428284964

지하철

In [77]:
df_GB_s= od_subway[od_subway['dest_hdong_cd'] == '5115058000.0' ]
df_GB_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination


In [78]:
# 지하철은 1 km당 약 3~5 kWh의 전력을 소비
#(전기 사용량 * 0.4781)
carbon_footprint_seoul_subway = (od_dist_avg_mean_subway * 4 * 0.4781) / 30
carbon_footprint_seoul_subway

3484.286255217904

철도

In [79]:
df_GB_t= od_train[od_train['dest_hdong_cd'] == '5115058000.0' ]
df_GB_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
476108,5115054000.0,5115058000.0,20230902,18:00,19:00,0.0,1.0,철도,여행,여행,61383.0,107.0,7.0,강릉시 초당동
295432,1168059000.0,5115058000.0,20230908,15:00,17:00,0.0,0.0,철도,귀가,여행,268303.0,153.0,24.0,강릉시 초당동
145861,1156058500.0,5115058000.0,20230923,11:00,16:00,1.0,0.0,철도,귀가,여행,244864.0,246.0,12.0,강릉시 초당동
547824,1123056000.0,5115058000.0,20230925,09:00,12:00,1.0,3.0,철도,여행,여행,632460.0,191.0,7.0,강릉시 초당동
112802,1129081000.0,5115058000.0,20230929,11:00,16:00,0.0,1.0,철도,귀가,여행,295330.0,284.0,7.0,강릉시 초당동
629114,1153052000.0,5115058000.0,20230929,12:00,16:00,0.0,1.0,철도,귀가,여행,713133.0,238.0,7.0,강릉시 초당동
31950,4121063300.0,5115058000.0,20230929,11:00,16:00,0.0,1.0,철도,기타,여행,507370.0,280.0,7.0,강릉시 초당동
23972,5113054200.0,5115058000.0,20230930,09:00,11:00,0.0,1.0,철도,귀가,기타,195997.0,122.0,7.0,강릉시 초당동
603716,4119074200.0,5115058000.0,20231001,12:00,16:00,0.0,1.0,철도,귀가,여행,354048.0,242.0,7.0,강릉시 초당동
397018,1165053100.0,5115058000.0,20231002,13:00,16:00,0.0,1.0,철도,여행,여행,427425.0,221.0,7.0,강릉시 초당동


In [80]:
df_gb_date_t=df_GB_t[(df_GB_t['date']>= 20231012) & (df_GB_t['date'] <= 20231015)]
df_gb_date_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
189744,1123056000.0,5115058000.0,20231015,10:00,12:00,0.0,1.0,철도,쇼핑여가,여행,339695.0,152.0,7.0,강릉시 초당동


In [81]:
# od_dist_avg 값들의 평균 계산
dist_gb_t = df_gb_date_t['od_dist_avg'].mean()
dist_gb_t

339695.0

In [82]:
# KTX는 주행 시 평균적으로 1km당 약 2.5 kWh에서 3 kWh의 전력 소비
#(전기 사용량 * 0.4781)
carbon_footprint_gb_train = (dist_gb_t * 2.5 * 0.4781) / (200*24)
carbon_footprint_gb_train

84.58759348958334

# 부산


In [83]:
df_b = od_car[od_car['dest_hdong_cd'] == '2635052000.0']
df_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
12834,2635051000.0,2635052000.0,20230901,09:00,10:00,0.0,0.0,차량,여행,여행,99250.0,94.0,26.0,부산광역시 해운대구 우2동
19202,2635065000.0,2635052000.0,20230901,19:00,21:00,1.0,0.0,차량,여행,여행,59380.0,100.0,10.0,부산광역시 해운대구 우2동
29899,2635053000.0,2635052000.0,20230901,15:00,16:00,1.0,0.0,차량,기타,기타,9012.0,23.0,10.0,부산광역시 해운대구 우2동
32217,2635053000.0,2635052000.0,20230901,17:00,17:00,1.0,5.0,차량,여행,여행,55541.0,32.0,8.0,부산광역시 해운대구 우2동
33877,2635051000.0,2635052000.0,20230901,19:00,19:00,1.0,3.0,차량,여행,여행,10129.0,18.0,8.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695284,2635051000.0,2635052000.0,20231015,11:00,12:00,1.0,3.0,차량,귀가,쇼핑여가,82411.0,41.0,6.0,부산광역시 해운대구 우2동
703324,2629058000.0,2635052000.0,20231015,17:00,18:00,1.0,2.0,차량,귀가,쇼핑여가,59930.0,38.0,5.0,부산광역시 해운대구 우2동
704100,2629057000.0,2635052000.0,20231015,12:00,13:00,1.0,1.0,차량,귀가,쇼핑여가,70986.0,55.0,5.0,부산광역시 해운대구 우2동
709966,2641057000.0,2635052000.0,20231015,11:00,11:00,1.0,2.0,차량,귀가,쇼핑여가,37356.0,43.0,5.0,부산광역시 해운대구 우2동


In [84]:
df_b_date=df_b[(df_b['date']>= 20231004) & (df_b['date'] <= 20231013)]
df_b_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
177165,2635065000.0,2635052000.0,20231004,13:00,13:00,0.0,3.0,차량,기타,쇼핑여가,5497.0,12.0,8.0,부산광역시 해운대구 우2동
179256,2650080000.0,2635052000.0,20231004,18:00,18:00,1.0,2.0,차량,여행,여행,9876.0,11.0,8.0,부산광역시 해운대구 우2동
192200,2650075000.0,2635052000.0,20231004,21:00,21:00,1.0,2.0,차량,여행,여행,8300.0,16.0,8.0,부산광역시 해운대구 우2동
193435,2647068000.0,2635052000.0,20231004,14:00,14:00,0.0,0.0,차량,귀가,기타,3585.0,14.0,13.0,부산광역시 해운대구 우2동
194350,2635051000.0,2635052000.0,20231004,14:00,15:00,0.0,3.0,차량,여행,여행,63711.0,45.0,8.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308922,2635065000.0,2635052000.0,20231013,19:00,19:00,1.0,1.0,차량,쇼핑여가,쇼핑여가,4982.0,6.0,5.0,부산광역시 해운대구 우2동
323503,2729058500.0,2635052000.0,20231013,09:00,12:00,1.0,2.0,차량,귀가,여행,319828.0,157.0,5.0,부산광역시 해운대구 우2동
332603,2638053000.0,2635052000.0,20231013,10:00,11:00,1.0,2.0,차량,귀가,쇼핑여가,17296.0,41.0,5.0,부산광역시 해운대구 우2동
338955,2635051000.0,2635052000.0,20231013,09:00,10:00,1.0,2.0,차량,여행,여행,97233.0,47.0,5.0,부산광역시 해운대구 우2동


In [156]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_b_date['dist_group'] = pd.cut(df_b_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_b_date['weight'] = df_b_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_b_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_b = np.average(df_b_date['od_dist_avg'], weights=df_b_date['weight'])
print("가중 평균:", weighted_avg_b)

가중 평균: 48238.322985152605


<ipython-input-156-e2ed338a8575>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date['dist_group'] = pd.cut(df_b_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-156-e2ed338a8575>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date['weight'] = df_b_date['dist_group'].map(group_weights)
<ipython-input-156-e2ed338a8575>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The

In [157]:
# od_dist_avg 값들의 평균 계산
dist_b = weighted_avg_b
dist_b

48238.322985152605

In [158]:
#휘발유로 통일_탄소발자국 공식 사용

carbon_footprint_busan = (dist_b / 16.04) * 2.097
carbon_footprint_busan

6306.46903365742

In [87]:
df_bb= od_bus[od_bus['dest_hdong_cd'] == '2635052000.0']
df_bb

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
5184,2635065000.0,2635052000.0,20230901,15:00,15:00,1.0,3.0,시내버스,귀가,쇼핑여가,9099.0,6.0,8.0,부산광역시 해운대구 우2동
57244,2635052500.0,2635052000.0,20230901,10:00,10:00,1.0,4.0,시내버스,귀가,쇼핑여가,10272.0,17.0,14.0,부산광역시 해운대구 우2동
85102,2635051000.0,2635052000.0,20230901,17:00,17:00,0.0,0.0,시내버스,여행,여행,45915.0,32.0,13.0,부산광역시 해운대구 우2동
92559,2635065000.0,2635052000.0,20230901,13:00,13:00,1.0,0.0,시내버스,귀가,쇼핑여가,9461.0,11.0,21.0,부산광역시 해운대구 우2동
148578,2635051000.0,2635052000.0,20230901,19:00,19:00,1.0,0.0,시내버스,여행,여행,14959.0,17.0,10.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607033,2635065000.0,2635052000.0,20231015,08:00,09:00,1.0,4.0,시내버스,귀가,기타,16497.0,18.0,6.0,부산광역시 해운대구 우2동
623535,2635052500.0,2635052000.0,20231015,10:00,11:00,0.0,5.0,시내버스,귀가,기타,25086.0,31.0,5.0,부산광역시 해운대구 우2동
651957,2635055400.0,2635052000.0,20231015,09:00,09:00,0.0,1.0,시내버스,귀가,기타,28364.0,25.0,5.0,부산광역시 해운대구 우2동
667263,2671025000.0,2635052000.0,20231015,15:00,16:00,1.0,2.0,시내버스,기타,쇼핑여가,41651.0,55.0,5.0,부산광역시 해운대구 우2동


In [88]:
df_b_date_b=df_bb[(df_bb['date']>= 20231004) & (df_bb['date'] <= 20231013)]
df_b_date_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
186426,2626057000.0,2635052000.0,20231004,08:00,09:00,1.0,0.0,시내버스,귀가,기타,10859.0,30.0,31.0,부산광역시 해운대구 우2동
190751,2635065000.0,2635052000.0,20231004,12:00,13:00,1.0,2.0,시내버스,여행,여행,34939.0,65.0,8.0,부산광역시 해운대구 우2동
266527,2635065000.0,2635052000.0,20231004,13:00,13:00,1.0,5.0,시내버스,귀가,쇼핑여가,8684.0,15.0,10.0,부산광역시 해운대구 우2동
315502,2650077000.0,2635052000.0,20231004,17:00,18:00,0.0,2.0,시내버스,여행,여행,12379.0,27.0,9.0,부산광역시 해운대구 우2동
524122,2635055100.0,2635052000.0,20231004,12:00,12:00,1.0,4.0,시내버스,귀가,쇼핑여가,21357.0,19.0,6.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167897,2635052500.0,2635052000.0,20231013,09:00,10:00,1.0,5.0,시내버스,귀가,쇼핑여가,13196.0,30.0,6.0,부산광역시 해운대구 우2동
167899,2635053000.0,2635052000.0,20231013,14:00,15:00,1.0,3.0,시내버스,쇼핑여가,쇼핑여가,35614.0,74.0,6.0,부산광역시 해운대구 우2동
241517,2635053000.0,2635052000.0,20231013,16:00,17:00,1.0,3.0,시내버스,귀가,쇼핑여가,24678.0,35.0,6.0,부산광역시 해운대구 우2동
299585,2635055200.0,2635052000.0,20231013,12:00,12:00,1.0,1.0,시내버스,기타,쇼핑여가,14733.0,23.0,5.0,부산광역시 해운대구 우2동


In [159]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_b_date_b['dist_group'] = pd.cut(df_b_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_b_date_b['weight'] = df_b_date_b['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_b_date_b['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_bb = np.average(df_b_date_b['od_dist_avg'], weights=df_b_date_b['weight'])
print("가중 평균:", weighted_avg_bb)

가중 평균: 32095.55519329296


<ipython-input-159-a12736532283>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date_b['dist_group'] = pd.cut(df_b_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-159-a12736532283>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date_b['weight'] = df_b_date_b['dist_group'].map(group_weights)
<ipython-input-159-a12736532283>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [160]:
# od_dist_avg 값들의 평균 계산
dist_b_b= weighted_avg_bb
dist_b_b

32095.55519329296

In [161]:
 #보통 시내버스의 CNG 소비율은 km당 0.35 kg에서 0.5 kg 사이
 # 시내버스는 CNG(천연가스)로 통일. 주행 거리 (km)×0.4kg/km×2.75kg CO2/kg   -> 공식 이상한듯
 # 시내버스 경유로 통일

carbon_footprint_busan_b = (dist_b_b/ 15.35) * 2.582
carbon_footprint_busan_b

5398.744202546086

지하철

In [91]:
df_s = od_subway[od_subway['dest_hdong_cd'] == '2635052000.0']
df_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
345303,2650076000.0,2635052000.0,20230901,09:00,09:00,0.0,2.0,지하철,귀가,기타,21416.0,19.0,6.0,부산광역시 해운대구 우2동
418090,2623060000.0,2635052000.0,20230901,14:00,14:00,1.0,3.0,지하철,쇼핑여가,쇼핑여가,36794.0,40.0,6.0,부산광역시 해운대구 우2동
427706,2629053000.0,2635052000.0,20230901,15:00,16:00,0.0,2.0,지하철,귀가,쇼핑여가,36377.0,35.0,6.0,부산광역시 해운대구 우2동
566436,2629053000.0,2635052000.0,20230901,11:00,12:00,0.0,2.0,지하철,쇼핑여가,쇼핑여가,18310.0,30.0,6.0,부산광역시 해운대구 우2동
578833,2653062000.0,2635052000.0,20230901,18:00,19:00,0.0,3.0,지하철,업무,쇼핑여가,34449.0,55.0,6.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640801,2641060000.0,2635052000.0,20231015,14:00,15:00,1.0,2.0,지하철,귀가,쇼핑여가,43409.0,61.0,5.0,부산광역시 해운대구 우2동
658815,2629053000.0,2635052000.0,20231015,13:00,13:00,1.0,2.0,지하철,기타,기타,16368.0,24.0,5.0,부산광역시 해운대구 우2동
659831,2635051000.0,2635052000.0,20231015,12:00,13:00,1.0,3.0,지하철,여행,여행,86961.0,67.0,6.0,부산광역시 해운대구 우2동
689070,2635053000.0,2635052000.0,20231015,16:00,16:00,0.0,3.0,지하철,귀가,쇼핑여가,11429.0,26.0,6.0,부산광역시 해운대구 우2동


In [92]:
df_b_date_s=df_s[(df_s['date']>= 20231004) & (df_s['date'] <= 20231013)]
df_b_date_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
190164,2635055200.0,2635052000.0,20231004,09:00,09:00,1.0,0.0,지하철,기타,기타,15227.0,30.0,10.0,부산광역시 해운대구 우2동
206700,2650066000.0,2635052000.0,20231004,16:00,17:00,1.0,0.0,지하철,귀가,쇼핑여가,45282.0,28.0,10.0,부산광역시 해운대구 우2동
271600,2635055200.0,2635052000.0,20231004,09:00,09:00,1.0,0.0,지하철,귀가,쇼핑여가,15497.0,35.0,31.0,부산광역시 해운대구 우2동
291906,2635055100.0,2635052000.0,20231004,08:00,09:00,1.0,0.0,지하철,귀가,쇼핑여가,15224.0,45.0,10.0,부산광역시 해운대구 우2동
396684,2635053000.0,2635052000.0,20231004,08:00,09:00,1.0,0.0,지하철,귀가,쇼핑여가,17208.0,47.0,21.0,부산광역시 해운대구 우2동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269953,2653066100.0,2635052000.0,20231013,08:00,09:00,0.0,1.0,지하철,기타,기타,52470.0,69.0,5.0,부산광역시 해운대구 우2동
295399,2635053000.0,2635052000.0,20231013,12:00,12:00,1.0,1.0,지하철,귀가,쇼핑여가,14933.0,29.0,5.0,부산광역시 해운대구 우2동
322415,2614064000.0,2635052000.0,20231013,08:00,09:00,1.0,2.0,지하철,귀가,기타,39456.0,53.0,5.0,부산광역시 해운대구 우2동
328417,2650079000.0,2635052000.0,20231013,17:00,17:00,1.0,2.0,지하철,귀가,기타,11100.0,20.0,5.0,부산광역시 해운대구 우2동


In [162]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_b_date_s['dist_group'] = pd.cut(df_b_date_s['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_b_date_s['weight'] = df_b_date_s['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_b_date_s['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_bs = np.average(df_b_date_s['od_dist_avg'], weights=df_b_date_s['weight'])
print("가중 평균:", weighted_avg_bs)

가중 평균: 33585.62910216718


<ipython-input-162-c12bbfaacd28>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date_s['dist_group'] = pd.cut(df_b_date_s['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-162-c12bbfaacd28>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date_s['weight'] = df_b_date_s['dist_group'].map(group_weights)
<ipython-input-162-c12bbfaacd28>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [163]:
dist_busan_s = weighted_avg_bs
dist_busan_s

33585.62910216718

In [164]:
# 지하철은 1 km당 약 3~5 kWh의 전력을 소비
#(전기 사용량 * 0.4781)
carbon_footprint_busan_subway = (dist_busan_s * 4 * 0.4781) / ( 30 * 24)
carbon_footprint_busan_subway

89.20716263192294

철도

In [95]:
df_bt= od_train[od_train['dest_hdong_cd'] == '2635052000.0']
df_bt

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
145672,3114054000.0,2635052000.0,20230923,15:00,18:00,0.0,0.0,철도,쇼핑여가,쇼핑여가,68685.0,128.0,13.0,부산광역시 해운대구 우2동
148334,3171025600.0,2635052000.0,20230923,13:00,14:00,1.0,0.0,철도,여행,여행,65499.0,90.0,10.0,부산광역시 해운대구 우2동
215347,2671025000.0,2635052000.0,20230930,15:00,16:00,1.0,0.0,철도,쇼핑여가,기타,81905.0,97.0,10.0,부산광역시 해운대구 우2동
232254,3171026200.0,2635052000.0,20231001,08:00,10:00,1.0,3.0,철도,귀가,쇼핑여가,152226.0,93.0,6.0,부산광역시 해운대구 우2동
70270,1117066000.0,2635052000.0,20231006,15:00,20:00,0.0,2.0,철도,기타,여행,900019.0,304.0,6.0,부산광역시 해운대구 우2동
473442,3120056000.0,2635052000.0,20231007,14:00,16:00,1.0,0.0,철도,기타,여행,86483.0,108.0,10.0,부산광역시 해운대구 우2동
498434,3120051000.0,2635052000.0,20231008,09:00,12:00,1.0,0.0,철도,귀가,쇼핑여가,413475.0,145.0,10.0,부산광역시 해운대구 우2동
349370,3171025600.0,2635052000.0,20231008,10:00,12:00,1.0,0.0,철도,귀가,기타,100844.0,121.0,10.0,부산광역시 해운대구 우2동
483869,3114059500.0,2635052000.0,20231015,11:00,13:00,0.0,0.0,철도,귀가,쇼핑여가,227785.0,151.0,13.0,부산광역시 해운대구 우2동


In [96]:
df_b_date_t=df_bt[(df_bt['date']>= 20231004) & (df_bt['date'] <= 20231013)]
df_b_date_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
70270,1117066000.0,2635052000.0,20231006,15:00,20:00,0.0,2.0,철도,기타,여행,900019.0,304.0,6.0,부산광역시 해운대구 우2동
473442,3120056000.0,2635052000.0,20231007,14:00,16:00,1.0,0.0,철도,기타,여행,86483.0,108.0,10.0,부산광역시 해운대구 우2동
498434,3120051000.0,2635052000.0,20231008,09:00,12:00,1.0,0.0,철도,귀가,쇼핑여가,413475.0,145.0,10.0,부산광역시 해운대구 우2동
349370,3171025600.0,2635052000.0,20231008,10:00,12:00,1.0,0.0,철도,귀가,기타,100844.0,121.0,10.0,부산광역시 해운대구 우2동


In [165]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_b_date_t['dist_group'] = pd.cut(df_b_date_t['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_b_date_t['weight'] = df_b_date_t['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_b_date_t['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_bt = np.average(df_b_date_t['od_dist_avg'], weights=df_b_date_t['weight'])
print("가중 평균:", weighted_avg_bt)

가중 평균: 427350.8461538462


<ipython-input-165-54d923593a5f>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date_t['dist_group'] = pd.cut(df_b_date_t['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-165-54d923593a5f>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_b_date_t['weight'] = df_b_date_t['dist_group'].map(group_weights)
<ipython-input-165-54d923593a5f>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [166]:
dist_busan_t = weighted_avg_bt
dist_busan_t

427350.8461538462

In [167]:
# KTX는 주행 시 평균적으로 1km당 약 2.5 kWh에서 3 kWh의 전력 소비
#(전기 사용량 * 0.4781)
carbon_footprint_seoul_train = (dist_busan_t * 2.5 * 0.4781) / (200 * 24)
carbon_footprint_seoul_train

106.41481226362181

#대전

In [99]:
df_d = od_car[od_car['dest_hdong_cd'] == '3020055000.0']
df_d

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
1363,3611055600.0,3020055000.0,20230901,15:00,15:00,0.0,0.0,차량,귀가,기타,43798.0,42.0,27.0,대전광역시 유성구 도룡동
17636,3020060000.0,3020055000.0,20230901,12:00,12:00,0.0,4.0,차량,여행,여행,8053.0,11.0,10.0,대전광역시 유성구 도룡동
19600,3020060000.0,3020055000.0,20230901,21:00,21:00,1.0,0.0,차량,기타,기타,1534.0,5.0,21.0,대전광역시 유성구 도룡동
26589,3020054000.0,3020055000.0,20230901,09:00,09:00,1.0,4.0,차량,귀가,기타,12216.0,16.0,12.0,대전광역시 유성구 도룡동
28863,3611052300.0,3020055000.0,20230901,15:00,17:00,1.0,0.0,차량,기타,여행,55351.0,115.0,21.0,대전광역시 유성구 도룡동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
700621,3017064000.0,3020055000.0,20231015,15:00,15:00,1.0,3.0,차량,기타,기타,11082.0,19.0,5.0,대전광역시 유성구 도룡동
701919,3020052700.0,3020055000.0,20231015,15:00,15:00,1.0,4.0,차량,귀가,기타,13946.0,30.0,5.0,대전광역시 유성구 도룡동
707780,3611053000.0,3020055000.0,20231015,16:00,17:00,1.0,3.0,차량,귀가,쇼핑여가,44496.0,49.0,5.0,대전광역시 유성구 도룡동
708038,3017065000.0,3020055000.0,20231015,14:00,15:00,1.0,4.0,차량,쇼핑여가,쇼핑여가,14900.0,40.0,5.0,대전광역시 유성구 도룡동


In [100]:
df_d_date=df_d[(df_d['date']>= 20231008) & (df_d['date'] <= 20231010)]
df_d_date

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
272689,3014074000.0,3020055000.0,20231008,18:00,19:00,1.0,4.0,차량,여행,여행,26308.0,50.0,7.0,대전광역시 유성구 도룡동
276077,3611051800.0,3020055000.0,20231008,11:00,11:00,1.0,0.0,차량,귀가,쇼핑여가,34328.0,33.0,21.0,대전광역시 유성구 도룡동
278053,3023060000.0,3020055000.0,20231008,13:00,13:00,1.0,4.0,차량,귀가,기타,9462.0,15.0,7.0,대전광역시 유성구 도룡동
283428,3011055200.0,3020055000.0,20231008,15:00,16:00,1.0,0.0,차량,여행,여행,32589.0,44.0,21.0,대전광역시 유성구 도룡동
287602,3014074000.0,3020055000.0,20231008,13:00,14:00,1.0,0.0,차량,여행,여행,7678.0,54.0,21.0,대전광역시 유성구 도룡동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124553,3020057000.0,3020055000.0,20231010,10:00,11:00,1.0,3.0,차량,귀가,쇼핑여가,14645.0,43.0,5.0,대전광역시 유성구 도룡동
131063,3014065500.0,3020055000.0,20231010,10:00,11:00,1.0,3.0,차량,쇼핑여가,쇼핑여가,10904.0,24.0,5.0,대전광역시 유성구 도룡동
168950,3017063000.0,3020055000.0,20231010,15:00,15:00,1.0,3.0,차량,쇼핑여가,쇼핑여가,14183.0,24.0,5.0,대전광역시 유성구 도룡동
181153,3017064000.0,3020055000.0,20231010,12:00,14:00,1.0,4.0,차량,귀가,기타,25329.0,96.0,5.0,대전광역시 유성구 도룡동


In [168]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_d_date['dist_group'] = pd.cut(df_d_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_d_date['weight'] = df_d_date['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_d_date['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_d = np.average(df_d_date['od_dist_avg'], weights=df_d_date['weight'])
print("가중 평균:", weighted_avg_d)

가중 평균: 54097.00563380281


<ipython-input-168-f87316ccfa5c>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_date['dist_group'] = pd.cut(df_d_date['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-168-f87316ccfa5c>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_date['weight'] = df_d_date['dist_group'].map(group_weights)
<ipython-input-168-f87316ccfa5c>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The

In [169]:
# od_dist_avg 값들의 평균 계산
dist_dae = weighted_avg_d
dist_dae

54097.00563380281

In [170]:
#휘발유로 통일_탄소발자국 공식 사용

carbon_footprint_dae = (dist_dae / 16.04) * 2.097
carbon_footprint_dae

7072.407781426714

시내버스

In [103]:
df_db = od_bus[od_bus['dest_hdong_cd'] == '3020055000.0']
df_db

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
98311,3020054000.0,3020055000.0,20230901,11:00,11:00,0.0,5.0,시내버스,업무,쇼핑여가,8539.0,15.0,12.0,대전광역시 유성구 도룡동
99999,3020060000.0,3020055000.0,20230901,09:00,09:00,1.0,3.0,시내버스,귀가,기타,11536.0,21.0,8.0,대전광역시 유성구 도룡동
122004,3017064000.0,3020055000.0,20230901,11:00,11:00,1.0,0.0,시내버스,귀가,쇼핑여가,4817.0,13.0,21.0,대전광역시 유성구 도룡동
129692,3020054000.0,3020055000.0,20230901,11:00,11:00,0.0,3.0,시내버스,업무,쇼핑여가,6842.0,8.0,19.0,대전광역시 유성구 도룡동
159232,3020057000.0,3020055000.0,20230901,18:00,18:00,1.0,1.0,시내버스,귀가,기타,15651.0,25.0,8.0,대전광역시 유성구 도룡동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
630772,3020054000.0,3020055000.0,20231015,14:00,15:00,1.0,4.0,시내버스,귀가,기타,62279.0,81.0,5.0,대전광역시 유성구 도룡동
635335,3020054000.0,3020055000.0,20231015,21:00,21:00,0.0,5.0,시내버스,귀가,기타,8954.0,13.0,5.0,대전광역시 유성구 도룡동
664547,3020057000.0,3020055000.0,20231015,10:00,10:00,1.0,3.0,시내버스,귀가,쇼핑여가,9738.0,14.0,5.0,대전광역시 유성구 도룡동
672703,3017064000.0,3020055000.0,20231015,11:00,12:00,1.0,3.0,시내버스,여행,여행,62930.0,64.0,5.0,대전광역시 유성구 도룡동


In [104]:
df_d_date_b=df_db[(df_db['date']>= 20231008) & (df_db['date'] <= 20231010)]
df_d_date_b

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
274158,3014067000.0,3020055000.0,20231008,10:00,10:00,0.0,0.0,시내버스,귀가,쇼핑여가,9508.0,26.0,27.0,대전광역시 유성구 도룡동
338077,3020053000.0,3020055000.0,20231008,11:00,11:00,0.0,1.0,시내버스,귀가,쇼핑여가,10175.0,19.0,9.0,대전광역시 유성구 도룡동
385528,3017063000.0,3020055000.0,20231008,10:00,10:00,1.0,4.0,시내버스,귀가,쇼핑여가,10099.0,13.0,10.0,대전광역시 유성구 도룡동
390272,3020060000.0,3020055000.0,20231008,17:00,17:00,1.0,0.0,시내버스,귀가,기타,12930.0,16.0,21.0,대전광역시 유성구 도룡동
406247,3020052700.0,3020055000.0,20231008,11:00,11:00,1.0,0.0,시내버스,귀가,기타,10197.0,20.0,21.0,대전광역시 유성구 도룡동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
710057,3017063000.0,3020055000.0,20231010,10:00,10:00,1.0,4.0,시내버스,귀가,쇼핑여가,11437.0,12.0,5.0,대전광역시 유성구 도룡동
19881,3017064000.0,3020055000.0,20231010,11:00,11:00,1.0,3.0,시내버스,귀가,쇼핑여가,9055.0,13.0,5.0,대전광역시 유성구 도룡동
90642,3017063000.0,3020055000.0,20231010,12:00,12:00,1.0,4.0,시내버스,쇼핑여가,쇼핑여가,14025.0,20.0,5.0,대전광역시 유성구 도룡동
134888,3020054000.0,3020055000.0,20231010,17:00,18:00,1.0,3.0,시내버스,업무,쇼핑여가,7344.0,17.0,5.0,대전광역시 유성구 도룡동


In [171]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_d_date_b['dist_group'] = pd.cut(df_d_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_d_date_b['weight'] = df_d_date_b['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_d_date_b['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_db = np.average(df_d_date_b['od_dist_avg'], weights=df_d_date_b['weight'])
print("가중 평균:", weighted_avg_db)

가중 평균: 13916.050632911394


<ipython-input-171-dabf077fc9d3>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_date_b['dist_group'] = pd.cut(df_d_date_b['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-171-dabf077fc9d3>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_date_b['weight'] = df_d_date_b['dist_group'].map(group_weights)
<ipython-input-171-dabf077fc9d3>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [172]:
# od_dist_avg 값들의 평균 계산
dist_dae_b = weighted_avg_db
dist_dae_b

13916.050632911394

In [173]:
 #보통 시내버스의 CNG 소비율은 km당 0.35 kg에서 0.5 kg 사이
 # 시내버스는 CNG(천연가스)로 통일. 주행 거리 (km)×0.4kg/km×2.75kg CO2/kg   -> 공식 이상한듯
 # 시내버스 경유로 통일

carbon_footprint_dae_bus = (dist_dae_b/ 15.35) * 2.582
carbon_footprint_dae_bus

2340.7975722591023

지하철

In [107]:
df_ds = od_subway[od_subway['dest_hdong_cd'] == '3020055000.0']
df_ds

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
238125,3011059000.0,3020055000.0,20230902,13:00,13:00,1.0,1.0,지하철,여행,여행,26990.0,47.0,8.0,대전광역시 유성구 도룡동
708979,3020053000.0,3020055000.0,20230903,10:00,11:00,0.0,0.0,지하철,여행,여행,42460.0,80.0,27.0,대전광역시 유성구 도룡동
279662,3020054600.0,3020055000.0,20230909,14:00,14:00,1.0,0.0,지하철,귀가,기타,26983.0,38.0,21.0,대전광역시 유성구 도룡동
338839,3014053500.0,3020055000.0,20230909,14:00,15:00,1.0,2.0,지하철,여행,여행,44055.0,70.0,6.0,대전광역시 유성구 도룡동
541762,3014053500.0,3020055000.0,20230909,15:00,16:00,1.0,2.0,지하철,여행,여행,27878.0,55.0,6.0,대전광역시 유성구 도룡동
102590,3611057000.0,3020055000.0,20230909,12:00,13:00,1.0,4.0,지하철,귀가,쇼핑여가,42252.0,77.0,5.0,대전광역시 유성구 도룡동
417055,3014072000.0,3020055000.0,20230909,14:00,16:00,1.0,0.0,지하철,귀가,기타,52347.0,88.0,21.0,대전광역시 유성구 도룡동
450582,3014053500.0,3020055000.0,20230909,14:00,15:00,1.0,1.0,지하철,여행,여행,33785.0,62.0,8.0,대전광역시 유성구 도룡동
437601,3017055500.0,3020055000.0,20230916,11:00,11:00,0.0,0.0,지하철,기타,쇼핑여가,8722.0,15.0,27.0,대전광역시 유성구 도룡동
253847,3017057500.0,3020055000.0,20230917,11:00,12:00,0.0,0.0,지하철,귀가,쇼핑여가,38394.0,66.0,27.0,대전광역시 유성구 도룡동


In [108]:
df_d_date_s=df_ds[(df_ds['date']>= 20231008) & (df_ds['date'] <= 20231010)]
df_d_date_s

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
136150,3020054600.0,3020055000.0,20231009,15:00,16:00,1.0,0.0,지하철,쇼핑여가,쇼핑여가,26801.0,57.0,21.0,대전광역시 유성구 도룡동
645183,3014067000.0,3020055000.0,20231009,11:00,12:00,1.0,0.0,지하철,귀가,쇼핑여가,31263.0,58.0,21.0,대전광역시 유성구 도룡동
11766,3014068000.0,3020055000.0,20231009,13:00,15:00,1.0,0.0,지하철,귀가,기타,34553.0,89.0,21.0,대전광역시 유성구 도룡동


In [174]:
import pandas as pd
import numpy as np

# 예시 데이터 (원본 데이터프레임 df_d 가 있다고 가정)
# od_dist_avg 값들을 5개의 구간으로 나누고, 임의로 가중치를 설정합니다.
bins = 5
group_names = ['group_A', 'group_B', 'group_C', 'group_D', 'group_E']

# 예시로 가중치를 임의로 설정 (각 그룹에 대한 가중치)
group_weights = {
    'group_A': 0.1,  # group_A에는 0.1의 가중치
    'group_B': 0.2,  # group_B에는 0.2의 가중치
    'group_C': 0.3,  # group_C에는 0.3의 가중치
    'group_D': 0.25, # group_D에는 0.25의 가중치
    'group_E': 0.15  # group_E에는 0.15의 가중치
}

# od_dist_avg 컬럼을 5개의 구간으로 나누고, 그룹 이름 할당
df_d_date_s['dist_group'] = pd.cut(df_d_date_s['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)

# dist_group에 임의로 설정한 가중치를 할당
df_d_date_s['weight'] = df_d_date_s['dist_group'].map(group_weights)

# NaN이 있는지 확인하고, NaN 값이 있으면 group_E로 채운다.
df_d_date_s['weight'].fillna(group_weights['group_E'], inplace=True)

# 가중 평균 계산
weighted_avg_ds = np.average(df_d_date_s['od_dist_avg'], weights=df_d_date_s['weight'])
print("가중 평균:", weighted_avg_ds)

가중 평균: 31349.0


<ipython-input-174-03725a2eb7e1>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_date_s['dist_group'] = pd.cut(df_d_date_s['od_dist_avg'], bins=bins, labels=group_names, include_lowest=True)
<ipython-input-174-03725a2eb7e1>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_date_s['weight'] = df_d_date_s['dist_group'].map(group_weights)
<ipython-input-174-03725a2eb7e1>:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace met

In [176]:
# od_dist_avg 값들의 평균 계산
dist_dae_s = weighted_avg_ds
dist_dae_s

31349.0

In [177]:
# 지하철은 1 km당 약 3~5 kWh의 전력을 소비
#(전기 사용량 * 0.4781)
carbon_footprint_dae_subway = (dist_dae_s * 4 * 0.4781) / (30*24)
carbon_footprint_dae_subway

83.26642722222223

철도

In [111]:
df_dt = od_train[od_train['dest_hdong_cd'] == '3020055000.0']
df_dt

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
184260,4374025000.0,3020055000.0,20230902,12:00,14:00,0.0,0.0,철도,귀가,쇼핑여가,92496.0,71.0,27.0,대전광역시 유성구 도룡동
257288,1150062000.0,3020055000.0,20230902,10:00,14:00,0.0,0.0,철도,귀가,여행,283507.0,214.0,27.0,대전광역시 유성구 도룡동
407312,1159060500.0,3020055000.0,20230902,09:00,12:00,0.0,0.0,철도,귀가,여행,238549.0,160.0,27.0,대전광역시 유성구 도룡동
686513,3011055100.0,3020055000.0,20230929,16:00,16:00,1.0,0.0,철도,쇼핑여가,기타,13223.0,24.0,21.0,대전광역시 유성구 도룡동
347004,1150063000.0,3020055000.0,20231006,13:00,17:00,1.0,0.0,철도,귀가,여행,375989.0,223.0,21.0,대전광역시 유성구 도룡동
229323,1117053000.0,3020055000.0,20231006,15:00,17:00,0.0,0.0,철도,귀가,여행,344481.0,102.0,27.0,대전광역시 유성구 도룡동
248134,1135061100.0,3020055000.0,20231013,17:00,21:00,1.0,0.0,철도,여행,기타,342636.0,207.0,21.0,대전광역시 유성구 도룡동


In [112]:
df_d_date_t=df_dt[(df_dt['date']>= 20231008) & (df_dt['date'] <= 20231010)]
df_d_date_t

,origin_hdong_cd,dest_hdong_cd,date,start_time,end_time,gender,age,modal,origin_purpose,dest_purpose,od_dist_avg,od_duration_avg,od_cnts,Destination
